## Iteration 2: Data Cleaning & Odds Feature Engineering

### Objective

Prepare a master dataset of football matches with a comprehensive set of match statistics and betting odds features to enable robust predictive modeling.

---

### Steps Performed

**1. Data Loading**
- Loaded the processed match data from `combinedWithOdds.csv`.

**2. Column Selection & Initial Cleaning**
- Selected relevant columns for match info, scores, odds, and handicap features.
- Dropped unnecessary columns (`Time`, `Attendance`, `HHW`, `AHW`, `HO`, `AO`, `Div`).

**3. Date Parsing & Filtering**
- Detected and unified multiple date formats.
- Parsed all dates; removed records before August 18, 2000.
- Added columns for season, year, month, and day of week.

**4. Odds Cleaning & Imputation**
- Converted all odds and score columns to numeric values.
- Dropped duplicate rows.
- Dropped rows missing essential bookmaker odds (`B365H`, `B365D`, `B365A`, `WHH`, `WHD`, `WHA`, `IWH`, `IWD`, `IWA`).
- Imputed missing values in core odds columns with the column mean.

**5. Target Variable Engineering**
- Computed key target variables for modeling:
  - `TotalGoals` (sum of home/away goals)
  - `GoalsOver2_5` (binary indicator for matches with more than 2.5 goals)
  - `BTTS` (both teams to score)
  - `Home_2plus` (home team scores 2+ goals)
  - `Away_2plus` (away team scores 2+ goals)

**6. Rolling Team Form Features**
- Sorted data chronologically.
- Computed rolling average goals for home and away teams.
- One-hot encoded home and away teams for ML compatibility.
- Engineered recent form features:
  - Rolling means of goals for, goals against, and points over the last 5 matches for each team.

**7. Advanced Feature Engineering**
- Calculated rolling recent points, goal difference, shots on target, and position differences.
- Imputed missing odds (`B365>2.5`, `B365<2.5`) with median values and flagged missingness.
- Engineered implied probabilities and odds ratios:
  - Implied probabilities (`B365H_prob`, `B365D_prob`, `B365A_prob`)
  - Normalized probabilities for each outcome.
  - Odds margin ratios (`OddsMargin`, `OverUnderRatio`, etc.)
- Added contextual indicators (weekend, early season).
- Computed referee aggression score (rolling sum of cards).

**8. Consensus Odds and Market Features**
- Aggregated bookmaker odds for each outcome (home, draw, away) with mean, min, max.
- Calculated odds spreads (max-min) for home, draw, away.
- Computed overround (bookmaker margin) for both single bookmaker and consensus odds.
- Aggregated consensus odds for over/under 2.5 goals, with implied probabilities.
- Engineered Asian Handicap consensus odds and market lines.
- Calculated home/away odds ratios for market strength comparison.

**9. Final Cleaning & Diagnostics**
- Replaced inf/nan values with column medians (numerics) or modes (categoricals).
- Dropped columns with >95% missing data (none remained).
- Verified that all columns have 0% missing data, and no columns exceeded 50% missingness.
- Saved the engineered DataFrame as `first_engineered_betting_features.csv`.

---

### Result

- **Final Data Shape:** 7,901 matches × 266 columns
- **Features:** Cleaned match stats, engineered targets, rolling team form, comprehensive consensus odds, market diagnostics, Asian Handicap, implied probabilities, odds ratios, and more.
- **Quality:** No columns have missing values; dataset is fully harmonized and ready for advanced modeling.
- **Export:** Saved as `../../data/processed/first_engineered_betting_features.csv`.

---

### Next Steps

- Utilize this feature-rich dataset for robust predictive modeling (Random Forest, XGBoost, calibration, etc.).
- Simulate business scenarios (betting, forecasting, strategy evaluation).
- Continue iterative improvements with new data sources and feature ideas.

---

In [1]:
import pandas as pd
import numpy as np
import re
import matplotlib.pyplot as plt
import seaborn as sns
pd.set_option('display.max_columns', None)

In [2]:
filePath = '../../data/processed/combinedWithOdds.csv'
df = pd.read_csv(filePath)

In [3]:
df

,Div,Date,HomeTeam,AwayTeam,FTHG,FTAG,FTR,HTHG,HTAG,HTR,Referee,HS,AS,HST,AST,HC,AC,HF,AF,HY,AY,HR,AR,B365H,B365D,B365A,VCH,VCD,VCA,BWH,BWD,BWA,GBH,GBD,GBA,IWH,IWD,IWA,LBH,LBD,LBA,SBH,SBD,SBA,SJH,SJD,SJA,WHH,WHD,WHA,Bb1X2,BbMxH,BbAvH,BbMxD,BbAvD,BbMxA,BbAvA,BbOU,BbMx>2.5,BbAv>2.5,BbMx<2.5,BbAv<2.5,BbAH,BbAHh,BbMxAHH,BbAvAHH,BbMxAHA,BbAvAHA,B365>2.5,B365<2.5,GB>2.5,GB<2.5,SOH,SOD,SOA,Time,B365AHH,B365AHA,PSH,PSD,PSA,P>2.5,P<2.5,PAHH,PAHA,MaxH,MaxD,MaxA,AvgH,AvgD,AvgA,Max>2.5,Max<2.5,Avg>2.5,Avg<2.5,MaxAHH,MaxAHA,AvgAHH,AvgAHA,AHh,BSH,BSD,BSA,Attendance,HHW,AHW,HO,AO,SYH,SYD,SYA,1XBH,1XBD,1XBA,BFH,BFD,BFA,BFEH,BFED,BFEA,B365AH,GBAHH,GBAHA,GBAH,LBAHH,LBAHA,LBAH
0,E0,19/08/06,Arsenal,Aston Villa,1.0,1.0,D,0.0,0.0,D,G Poll,19.0,5.0,11.0,3.0,18.0,1.0,10.0,19.0,1.0,2.0,0.0,0.0,1.28,4.50,13.00,1.30,4.50,11.00,1.25,5.00,10.00,1.27,5.00,11.00,1.30,4.40,8.50,1.29,4.00,10.00,1.28,4.75,11.00,1.29,4.50,10.00,1.28,4.50,8.00,47.0,1.33,1.27,5.50,4.82,13.25,10.72,39.0,1.83,1.75,2.12,2.01,28.0,-1.50,2.09,1.98,1.90,1.85,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,E0,19/08/06,Bolton,Tottenham,2.0,0.0,H,2.0,0.0,H,P Dowd,8.0,9.0,6.0,6.0,6.0,3.0,19.0,22.0,0.0,1.0,0.0,0.0,2.62,3.20,2.60,2.65,3.25,2.50,2.60,3.10,2.55,2.70,3.25,2.55,2.50,3.10,2.60,2.50,3.20,2.50,2.70,3.20,2.50,2.50,3.20,2.60,2.60,3.10,2.40,47.0,2.90,2.66,3.32,3.15,2.70,2.53,39.0,2.30,2.17,1.75,1.63,27.0,0.00,2.10,1.96,1.92,1.83,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,E0,19/08/06,Everton,Watford,2.0,1.0,H,1.0,0.0,H,P Walton,6.0,12.0,2.0,8.0,0.0,6.0,12.0,13.0,2.0,2.0,0.0,0.0,1.66,3.40,5.50,1.60,3.50,5.50,1.65,3.40,5.00,1.65,3.50,5.50,1.70,3.40,4.30,1.62,3.25,5.00,1.62,3.40,6.00,1.57,3.50,5.50,1.57,3.30,5.50,46.0,1.73,1.63,3.70,3.46,6.00,5.35,39.0,2.20,2.07,1.76,1.69,26.0,-0.75,2.00,1.94,1.96,1.93,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,E0,19/08/06,Newcastle,Wigan,2.0,1.0,H,1.0,0.0,H,M Atkinson,9.0,14.0,8.0,9.0,4.0,11.0,17.0,20.0,1.0,2.0,0.0,0.0,1.72,3.50,4.75,1.65,3.50,5.00,1.70,3.35,4.70,1.70,3.50,5.00,1.70,3.40,4.30,1.67,3.20,5.00,1.67,3.40,5.25,1.62,3.50,5.00,1.72,3.20,4.33,44.0,1.87,1.71,3.55,3.39,5.25,4.72,39.0,2.25,2.12,1.75,1.66,26.0,-0.75,2.18,2.06,1.93,1.81,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,E0,19/08/06,Portsmouth,Blackburn,3.0,0.0,H,1.0,0.0,H,A Wiley,21.0,9.0,16.0,7.0,6.0,2.0,20.0,17.0,2.0,1.0,0.0,2.0,2.30,3.10,3.10,2.40,3.25,2.75,2.45,3.10,2.70,2.40,3.20,2.90,2.50,3.10,2.60,2.25,3.20,2.75,2.38,3.20,2.88,2.25,3.25,2.88,2.20,3.10,2.87,46.0,2.51,2.37,3.33,3.20,3.20,2.83,39.0,2.26,2.15,1.69,1.64,27.0,0.00,1.86,1.75,2.20,2.06,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,

In [4]:
columns_to_keep = [
    'Div', 'Date', 'Time', 'HomeTeam', 'AwayTeam', 'FTHG', 'FTAG', 'FTR',
    'HTHG', 'HTAG', 'HTR', 'Attendance', 'Referee', 'HS', 'AS', 'HST', 'AST',
    'HHW', 'AHW', 'HC', 'AC', 'HF', 'AF', 'HFKC', 'AFKC', 'HO', 'AO', 'HY', 'AY',
    'HR', 'AR', '1XBH', '1XBD', '1XBA', 'B365H', 'B365D', 'B365A', 'B365>2.5', 'B365<2.5', 'B365AHH', 'B365AHA', 'B365AH',
    'BFH', 'BFD', 'BFA', 'BFEH', 'BFED', 'BFEA', 'BFDH', 'BFDD', 'BFDA',
    'BMGMH', 'BMGMD', 'BMGMA', 'BVH', 'BVD', 'BVA', 'VCH', 'VCD', 'VCA',
    'BSH', 'BSD', 'BSA', 'BWH', 'BWD', 'BWA', 'CLH', 'CLD', 'CLA',
    'GBH', 'GBD', 'GBA', 'GB>2.5', 'GB<2.5', 'GBAHH', 'GBAHA', 'GBAH',
    'IWH', 'IWD', 'IWA', 'LBH', 'LBD', 'LBA', 'LBAHH', 'LBAHA', 'LBAH',
    'PSH', 'PH', 'PSD', 'PD', 'PSA', 'PA', 'P>2.5', 'P<2.5', 'PAHH', 'PAHA',
    'SOH', 'SOD', 'SOA', 'SBH', 'SBD', 'SBA', 'SJH', 'SJD', 'SJA',
    'SYH', 'SYD', 'SYA', 'WHH', 'WHD', 'WHA',
    'Bb1X2', 'BbMxH', 'BbAvH', 'BbMxD', 'BbAvD', 'BbMxA', 'BbAvA',
    'BbOU', 'BbMx>2.5', 'BbAv>2.5', 'BbMx<2.5', 'BbAv<2.5',
    'BbAH', 'BbAHh', 'BbMxAHH', 'BbAvAHH', 'BbMxAHA', 'BbAvAHA',
    'MaxH', 'MaxD', 'MaxA', 'AvgH', 'AvgD', 'AvgA',
    'Max>2.5', 'Max<2.5', 'Avg>2.5', 'Avg<2.5',
    'MaxAHH', 'MaxAHA', 'AvgAHH', 'AvgAHA', 'AHh'
]

df = df[[col for col in columns_to_keep if col in df.columns]]

In [5]:
drop_cols = ['Time', 'Attendance', 'HHW', 'AHW', 'HO', 'AO', 'Div']
df = df.drop(columns=[c for c in drop_cols if c in df.columns], errors='ignore')

In [6]:
def date_format_type(date_str):
    if not isinstance(date_str, str):
        return "not_a_string"
    patterns = {
        "%d/%m/%y": r"^\d{2}/\d{2}/\d{2}$",
        "%d/%m/%Y": r"^\d{2}/\d{2}/\d{4}$",
        "%Y-%m-%d": r"^\d{4}-\d{2}-\d{2}$",
        "%m-%d-%Y": r"^\d{2}-\d{2}-\d{4}$",
        "%Y/%m/%d": r"^\d{4}/\d{2}/\d{2}$",
    }
    for fmt, pat in patterns.items():
        if re.match(pat, date_str):
            return fmt
    return "unknown"

In [7]:
df['DateFormat'] = df['Date'].apply(date_format_type)
def parse_dates(row):
    date_str = row['Date']
    if isinstance(date_str, str):
        try:
            return pd.to_datetime(date_str, format='%d/%m/%y')
        except ValueError:
            try:
                return pd.to_datetime(date_str, format='%d/%m/%Y')
            except ValueError:
                return pd.NaT
    else:
        return pd.NaT

df['Date'] = df.apply(parse_dates, axis=1)
df = df.drop(columns=['DateFormat'], errors='ignore')
df = df[df['Date'] >= pd.Timestamp('2000-08-18')]
df = df.reset_index(drop=True)

def get_season(date):
    if pd.isnull(date):
        return np.nan
    year = date.year
    month = date.month
    if month >= 8: 
        return f"{year}-{str(year+1)[-2:]}"
    else:
        return f"{year-1}-{str(year)[-2:]}"
df['Season'] = df['Date'].apply(get_season)
df['Year'] = df['Date'].dt.year
df['Month'] = df['Date'].dt.month
df['DayOfWeek'] = df['Date'].dt.dayofweek

In [8]:
df

,Date,HomeTeam,AwayTeam,FTHG,FTAG,FTR,HTHG,HTAG,HTR,Referee,HS,AS,HST,AST,HC,AC,HF,AF,HY,AY,HR,AR,1XBH,1XBD,1XBA,B365H,B365D,B365A,B365>2.5,B365<2.5,B365AHH,B365AHA,B365AH,BFH,BFD,BFA,BFEH,BFED,BFEA,VCH,VCD,VCA,BSH,BSD,BSA,BWH,BWD,BWA,GBH,GBD,GBA,GB>2.5,GB<2.5,GBAHH,GBAHA,GBAH,IWH,IWD,IWA,LBH,LBD,LBA,LBAHH,LBAHA,LBAH,PSH,PSD,PSA,P>2.5,P<2.5,PAHH,PAHA,SOH,SOD,SOA,SBH,SBD,SBA,SJH,SJD,SJA,SYH,SYD,SYA,WHH,WHD,WHA,Bb1X2,BbMxH,BbAvH,BbMxD,BbAvD,BbMxA,BbAvA,BbOU,BbMx>2.5,BbAv>2.5,BbMx<2.5,BbAv<2.5,BbAH,BbAHh,BbMxAHH,BbAvAHH,BbMxAHA,BbAvAHA,MaxH,MaxD,MaxA,AvgH,AvgD,AvgA,Max>2.5,Max<2.5,Avg>2.5,Avg<2.5,MaxAHH,MaxAHA,AvgAHH,AvgAHA,AHh,Season,Year,Month,DayOfWeek
0,2006-08-19,Arsenal,Aston Villa,1.0,1.0,D,0.0,0.0,D,G Poll,19.0,5.0,11.0,3.0,18.0,1.0,10.0,19.0,1.0,2.0,0.0,0.0,NaN,NaN,NaN,1.28,4.50,13.00,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1.30,4.50,11.00,NaN,NaN,NaN,1.25,5.00,10.00,1.27,5.00,11.00,NaN,NaN,NaN,NaN,NaN,1.30,4.40,8.50,1.29,4.00,10.00,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1.28,4.75,11.00,1.29,4.50,10.00,NaN,NaN,NaN,1.28,4.50,8.00,47.0,1.33,1.27,5.50,4.82,13.25,10.72,39.0,1.83,1.75,2.12,2.01,28.0,-1.50,2.09,1.98,1.90,1.85,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2006-07,2006,8,5
1,2006-08-19,Bolton,Tottenham,2.0,0.0,H,2.0,0.0,H,P Dowd,8.0,9.0,6.0,6.0,6.0,3.0,19.0,22.0,0.0,1.0,0.0,0.0,NaN,NaN,NaN,2.62,3.20,2.60,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2.65,3.25,2.50,NaN,NaN,NaN,2.60,3.10,2.55,2.70,3.25,2.55,NaN,NaN,NaN,NaN,NaN,2.50,3.10,2.60,2.50,3.20,2.50,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2.70,3.20,2.50,2.50,3.20,2.60,NaN,NaN,NaN,2.60,3.10,2.40,47.0,2.90,2.66,3.32,3.15,2.70,2.53,39.0,2.30,2.17,1.75,1.63,27.0,0.00,2.10,1.96,1.92,1.83,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2006-07,2006,8,5
2,2006-08-19,Everton,Watford,2.0,1.0,H,1.0,0.0,H,P Walton,6.0,12.0,2.0,8.0,0.0,6.0,12.0,13.0,2.0,2.0,0.0,0.0,NaN,NaN,NaN,1.66,3.40,5.50,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1.60,3.50,5.50,NaN,NaN,NaN,1.65,3.40,5.00,1.65,3.50,5.50,NaN,NaN,NaN,NaN,NaN,1.70,3.40,4.30,1.62,3.25,5.00,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1.62,3.40,6.00,1.57,3.50,5.50,NaN,NaN,NaN,1.57,3.30,5.50,46.0,1.73,1.63,3.70,3.46,6.00,5.35,39.0,2.20,2.07,1.76,1.69,26.0,-0.75,2.00,1.94,1.96,1.93,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2006-07,2006,8,5
3,2006-08-19,Newcastle,Wigan,2.0,1.0,H,1.0,0.0,H,M Atkinson,9.0,14.0,8.0,9.0,4.0,11.0,17.0,20.0,1.0,2.0,0.0,0.0,NaN,NaN,NaN,1.72,3.50,4.75,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1.65,3.50,5.00,NaN,NaN,NaN,1.70,3.35,4.70,1.70,3.50,5.00,NaN,NaN,NaN,NaN,NaN,1.70,3.40,4.30,1.67,3.20,5.00,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1.67,3.40,5.25,1.62,3.50,5.00,NaN,NaN,NaN,1.72,3.20,4.33,44.0,1.87,1.71,3.55,3.39,5.25,4.72,39.0,2.25,2.12,1.75,1.66,26.0,-0.75,2.18,2.06,1.93,1.81,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2006-07,2006,8,5
4,2006-08-19,Portsmouth,Blackburn,3.0,0.0,H,1.0,0.0,H,A Wiley,21.0,9.0,16.0,7.0,6.0,2.0,20.0,17.0,2.0,1.0,0.0,2.0,NaN,NaN,NaN,2.30,3.10,3.10,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2.40,3.25,2.75,NaN,NaN,NaN,2.45,3.10,2.70,2.40,3.20,2.90,NaN,NaN,NaN,NaN,NaN,2.50,3.10,2.60,2.25,3.20,2.75,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2.38,3.20,2.88,2.25,3.25,2.88,NaN,NaN,NaN,2.20,3.10,2.87,46.0,2.51,2.37,3.33,3.20,3.20,2.83,39.0,2.26,2.15,1.69,1.64,27.0,0.00,1.86,1.75,2.20,2.06,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2006-07,2006,8,5
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
94

In [9]:
odds_cols = [
    'B365H', 'B365D', 'B365A', 'B365>2.5', 'B365<2.5', 'B365AHH', 'B365AHA', 'B365AH',
    'IWH', 'IWD', 'IWA',
    'WHH', 'WHD', 'WHA',
    'PSH', 'PSD', 'PSA', 'PH', 'PD', 'PA', 'P>2.5', 'P<2.5', 'PAHH', 'PAHA',
    'LBH', 'LBD', 'LBA', 'LBAHH', 'LBAHA', 'LBAH',
    'GBH', 'GBD', 'GBA', 'GB>2.5', 'GB<2.5', 'GBAHH', 'GBAHA', 'GBAH',
    'BVH', 'BVD', 'BVA', 'VCH', 'VCD', 'VCA',
    '1XBH', '1XBD', '1XBA',
    'BWH', 'BWD', 'BWA',
    'SOH', 'SOD', 'SOA',
    'SBH', 'SBD', 'SBA',
    'CLH', 'CLD', 'CLA',
    'BMGMH', 'BMGMD', 'BMGMA',
    'BFDH', 'BFDD', 'BFDA',
    'BFH', 'BFD', 'BFA', 'BFEH', 'BFED', 'BFEA',
    'SYH', 'SYD', 'SYA',
    'SJH', 'SJD', 'SJA',
    'BSH', 'BSD', 'BSA',
    'BbMxH', 'BbAvH', 'BbMxD', 'BbAvD', 'BbMxA', 'BbAvA', 'BbOU', 'BbMx>2.5', 'BbAv>2.5', 'BbMx<2.5', 'BbAv<2.5',
    'BbAH', 'BbAHh', 'BbMxAHH', 'BbAvAHH', 'BbMxAHA', 'BbAvAHA',
    'MaxH', 'MaxD', 'MaxA', 'AvgH', 'AvgD', 'AvgA',
    'Max>2.5', 'Max<2.5', 'Avg>2.5', 'Avg<2.5',
    'MaxAHH', 'MaxAHA', 'AvgAHH', 'AvgAHA', 'AHh'
]

score_cols = [
    'FTHG', 'FTAG', 'HTHG', 'HTAG', 'HS', 'AS', 'HST', 'AST',  
    'HC', 'AC', 'HF', 'AF', 'HY', 'AY', 'HR', 'AR'
]

In [10]:
for col in odds_cols + score_cols:
    if col in df.columns:
        df[col] = pd.to_numeric(df[col], errors='coerce')

df = df.drop_duplicates()

In [11]:
df

,Date,HomeTeam,AwayTeam,FTHG,FTAG,FTR,HTHG,HTAG,HTR,Referee,HS,AS,HST,AST,HC,AC,HF,AF,HY,AY,HR,AR,1XBH,1XBD,1XBA,B365H,B365D,B365A,B365>2.5,B365<2.5,B365AHH,B365AHA,B365AH,BFH,BFD,BFA,BFEH,BFED,BFEA,VCH,VCD,VCA,BSH,BSD,BSA,BWH,BWD,BWA,GBH,GBD,GBA,GB>2.5,GB<2.5,GBAHH,GBAHA,GBAH,IWH,IWD,IWA,LBH,LBD,LBA,LBAHH,LBAHA,LBAH,PSH,PSD,PSA,P>2.5,P<2.5,PAHH,PAHA,SOH,SOD,SOA,SBH,SBD,SBA,SJH,SJD,SJA,SYH,SYD,SYA,WHH,WHD,WHA,Bb1X2,BbMxH,BbAvH,BbMxD,BbAvD,BbMxA,BbAvA,BbOU,BbMx>2.5,BbAv>2.5,BbMx<2.5,BbAv<2.5,BbAH,BbAHh,BbMxAHH,BbAvAHH,BbMxAHA,BbAvAHA,MaxH,MaxD,MaxA,AvgH,AvgD,AvgA,Max>2.5,Max<2.5,Avg>2.5,Avg<2.5,MaxAHH,MaxAHA,AvgAHH,AvgAHA,AHh,Season,Year,Month,DayOfWeek
0,2006-08-19,Arsenal,Aston Villa,1.0,1.0,D,0.0,0.0,D,G Poll,19.0,5.0,11.0,3.0,18.0,1.0,10.0,19.0,1.0,2.0,0.0,0.0,NaN,NaN,NaN,1.28,4.50,13.00,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1.30,4.50,11.00,NaN,NaN,NaN,1.25,5.00,10.00,1.27,5.00,11.00,NaN,NaN,NaN,NaN,NaN,1.30,4.40,8.50,1.29,4.00,10.00,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1.28,4.75,11.00,1.29,4.50,10.00,NaN,NaN,NaN,1.28,4.50,8.00,47.0,1.33,1.27,5.50,4.82,13.25,10.72,39.0,1.83,1.75,2.12,2.01,28.0,-1.50,2.09,1.98,1.90,1.85,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2006-07,2006,8,5
1,2006-08-19,Bolton,Tottenham,2.0,0.0,H,2.0,0.0,H,P Dowd,8.0,9.0,6.0,6.0,6.0,3.0,19.0,22.0,0.0,1.0,0.0,0.0,NaN,NaN,NaN,2.62,3.20,2.60,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2.65,3.25,2.50,NaN,NaN,NaN,2.60,3.10,2.55,2.70,3.25,2.55,NaN,NaN,NaN,NaN,NaN,2.50,3.10,2.60,2.50,3.20,2.50,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2.70,3.20,2.50,2.50,3.20,2.60,NaN,NaN,NaN,2.60,3.10,2.40,47.0,2.90,2.66,3.32,3.15,2.70,2.53,39.0,2.30,2.17,1.75,1.63,27.0,0.00,2.10,1.96,1.92,1.83,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2006-07,2006,8,5
2,2006-08-19,Everton,Watford,2.0,1.0,H,1.0,0.0,H,P Walton,6.0,12.0,2.0,8.0,0.0,6.0,12.0,13.0,2.0,2.0,0.0,0.0,NaN,NaN,NaN,1.66,3.40,5.50,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1.60,3.50,5.50,NaN,NaN,NaN,1.65,3.40,5.00,1.65,3.50,5.50,NaN,NaN,NaN,NaN,NaN,1.70,3.40,4.30,1.62,3.25,5.00,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1.62,3.40,6.00,1.57,3.50,5.50,NaN,NaN,NaN,1.57,3.30,5.50,46.0,1.73,1.63,3.70,3.46,6.00,5.35,39.0,2.20,2.07,1.76,1.69,26.0,-0.75,2.00,1.94,1.96,1.93,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2006-07,2006,8,5
3,2006-08-19,Newcastle,Wigan,2.0,1.0,H,1.0,0.0,H,M Atkinson,9.0,14.0,8.0,9.0,4.0,11.0,17.0,20.0,1.0,2.0,0.0,0.0,NaN,NaN,NaN,1.72,3.50,4.75,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1.65,3.50,5.00,NaN,NaN,NaN,1.70,3.35,4.70,1.70,3.50,5.00,NaN,NaN,NaN,NaN,NaN,1.70,3.40,4.30,1.67,3.20,5.00,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1.67,3.40,5.25,1.62,3.50,5.00,NaN,NaN,NaN,1.72,3.20,4.33,44.0,1.87,1.71,3.55,3.39,5.25,4.72,39.0,2.25,2.12,1.75,1.66,26.0,-0.75,2.18,2.06,1.93,1.81,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2006-07,2006,8,5
4,2006-08-19,Portsmouth,Blackburn,3.0,0.0,H,1.0,0.0,H,A Wiley,21.0,9.0,16.0,7.0,6.0,2.0,20.0,17.0,2.0,1.0,0.0,2.0,NaN,NaN,NaN,2.30,3.10,3.10,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2.40,3.25,2.75,NaN,NaN,NaN,2.45,3.10,2.70,2.40,3.20,2.90,NaN,NaN,NaN,NaN,NaN,2.50,3.10,2.60,2.25,3.20,2.75,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2.38,3.20,2.88,2.25,3.25,2.88,NaN,NaN,NaN,2.20,3.10,2.87,46.0,2.51,2.37,3.33,3.20,3.20,2.83,39.0,2.26,2.15,1.69,1.64,27.0,0.00,1.86,1.75,2.20,2.06,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2006-07,2006,8,5
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
94

In [12]:
essential_odds = ['B365H', 'B365D', 'B365A', 'WHH', 'WHD', 'WHA', 'IWH', 'IWD', 'IWA']

core_odds = [
    'B365H', 'B365D', 'B365A', 'B365>2.5', 'B365<2.5', 'B365AHH', 'B365AHA', 'B365AH',
    'IWH', 'IWD', 'IWA',
    'WHH', 'WHD', 'WHA',
    'PSH', 'PSD', 'PSA', 'PH', 'PD', 'PA', 'P>2.5', 'P<2.5', 'PAHH', 'PAHA',
    'LBH', 'LBD', 'LBA', 'LBAHH', 'LBAHA', 'LBAH',
    'GBH', 'GBD', 'GBA', 'GB>2.5', 'GB<2.5', 'GBAHH', 'GBAHA', 'GBAH',
    'BVH', 'BVD', 'BVA', 'VCH', 'VCD', 'VCA',
    '1XBH', '1XBD', '1XBA',
    'BWH', 'BWD', 'BWA',
    'SOH', 'SOD', 'SOA',
    'SBH', 'SBD', 'SBA',
    'CLH', 'CLD', 'CLA',
    'BMGMH', 'BMGMD', 'BMGMA',
    'BFDH', 'BFDD', 'BFDA',
    'BFH', 'BFD', 'BFA', 'BFEH', 'BFED', 'BFEA',
    'SYH', 'SYD', 'SYA',
    'SJH', 'SJD', 'SJA',
    'BSH', 'BSD', 'BSA',
    'BbMxH', 'BbAvH', 'BbMxD', 'BbAvD', 'BbMxA', 'BbAvA', 'BbOU', 'BbMx>2.5', 'BbAv>2.5', 'BbMx<2.5', 'BbAv<2.5',
    'BbAH', 'BbAHh', 'BbMxAHH', 'BbAvAHH', 'BbMxAHA', 'BbAvAHA',
    'MaxH', 'MaxD', 'MaxA', 'AvgH', 'AvgD', 'AvgA',
    'Max>2.5', 'Max<2.5', 'Avg>2.5', 'Avg<2.5',
    'MaxAHH', 'MaxAHA', 'AvgAHH', 'AvgAHA', 'AHh'
]

def clean_odds_dataframe(df, essential_odds, core_odds, fill_method='mean'):
    essentials = [col for col in essential_odds if col in df.columns]
    df = df.dropna(subset=essentials).reset_index(drop=True)
   
    for col in core_odds:
        if col in df.columns:
            if fill_method == 'mean':
                fill_value = df[col].mean()
            elif fill_method == 'median':
                fill_value = df[col].median()
            elif fill_method == 'zero':
                fill_value = 0
            elif fill_method == 'negone':
                fill_value = -1
            else:
                fill_value = None
            if fill_value is not None:
                df[col] = df[col].fillna(fill_value)
    return df

df = clean_odds_dataframe(df, essential_odds, core_odds, fill_method='mean')
df = df.reset_index(drop=True)

In [13]:
df

,Date,HomeTeam,AwayTeam,FTHG,FTAG,FTR,HTHG,HTAG,HTR,Referee,HS,AS,HST,AST,HC,AC,HF,AF,HY,AY,HR,AR,1XBH,1XBD,1XBA,B365H,B365D,B365A,B365>2.5,B365<2.5,B365AHH,B365AHA,B365AH,BFH,BFD,BFA,BFEH,BFED,BFEA,VCH,VCD,VCA,BSH,BSD,BSA,BWH,BWD,BWA,GBH,GBD,GBA,GB>2.5,GB<2.5,GBAHH,GBAHA,GBAH,IWH,IWD,IWA,LBH,LBD,LBA,LBAHH,LBAHA,LBAH,PSH,PSD,PSA,P>2.5,P<2.5,PAHH,PAHA,SOH,SOD,SOA,SBH,SBD,SBA,SJH,SJD,SJA,SYH,SYD,SYA,WHH,WHD,WHA,Bb1X2,BbMxH,BbAvH,BbMxD,BbAvD,BbMxA,BbAvA,BbOU,BbMx>2.5,BbAv>2.5,BbMx<2.5,BbAv<2.5,BbAH,BbAHh,BbMxAHH,BbAvAHH,BbMxAHA,BbAvAHA,MaxH,MaxD,MaxA,AvgH,AvgD,AvgA,Max>2.5,Max<2.5,Avg>2.5,Avg<2.5,MaxAHH,MaxAHA,AvgAHH,AvgAHA,AHh,Season,Year,Month,DayOfWeek
0,2006-08-19,Arsenal,Aston Villa,1.0,1.0,D,0.0,0.0,D,G Poll,19.0,5.0,11.0,3.0,18.0,1.0,10.0,19.0,1.0,2.0,0.0,0.0,NaN,NaN,NaN,1.28,4.50,13.00,1.867394,2.013934,1.953053,1.950898,-0.336141,NaN,NaN,NaN,NaN,NaN,NaN,1.30,4.50,11.00,2.624982,3.753932,4.749475,1.25,5.00,10.00,1.270000,5.000000,11.000000,1.847198,1.837633,1.89003,1.914697,-0.323485,1.30,4.40,8.50,1.290000,4.000000,10.000000,1.931357,1.914883,-0.349454,2.958986,4.266214,4.992703,1.856226,2.157956,1.967555,1.955565,2.31698,3.468319,4.037422,1.280000,4.750000,11.000000,1.290000,4.500000,10.000000,NaN,NaN,NaN,1.28,4.50,8.00,47.0,1.330000,1.270000,5.500000,4.820000,13.25000,10.720000,39.0000,1.830000,1.750000,2.120000,2.010000,28.00000,-1.500000,2.090000,1.98000,1.900000,1.850000,3.1928,4.493388,5.08369,3.003213,4.243009,4.610582,1.890303,2.207561,1.821665,2.115279,1.995425,1.990041,1.940827,1.934651,-0.260914,2006-07,2006,8,5
1,2006-08-19,Bolton,Tottenham,2.0,0.0,H,2.0,0.0,H,P Dowd,8.0,9.0,6.0,6.0,6.0,3.0,19.0,22.0,0.0,1.0,0.0,0.0,NaN,NaN,NaN,2.62,3.20,2.60,1.867394,2.013934,1.953053,1.950898,-0.336141,NaN,NaN,NaN,NaN,NaN,NaN,2.65,3.25,2.50,2.624982,3.753932,4.749475,2.60,3.10,2.55,2.700000,3.250000,2.550000,1.847198,1.837633,1.89003,1.914697,-0.323485,2.50,3.10,2.60,2.500000,3.200000,2.500000,1.931357,1.914883,-0.349454,2.958986,4.266214,4.992703,1.856226,2.157956,1.967555,1.955565,2.31698,3.468319,4.037422,2.700000,3.200000,2.500000,2.500000,3.200000,2.600000,NaN,NaN,NaN,2.60,3.10,2.40,47.0,2.900000,2.660000,3.320000,3.150000,2.70000,2.530000,39.0000,2.300000,2.170000,1.750000,1.630000,27.00000,0.000000,2.100000,1.96000,1.920000,1.830000,3.1928,4.493388,5.08369,3.003213,4.243009,4.610582,1.890303,2.207561,1.821665,2.115279,1.995425,1.990041,1.940827,1.934651,-0.260914,2006-07,2006,8,5
2,2006-08-19,Everton,Watford,2.0,1.0,H,1.0,0.0,H,P Walton,6.0,12.0,2.0,8.0,0.0,6.0,12.0,13.0,2.0,2.0,0.0,0.0,NaN,NaN,NaN,1.66,3.40,5.50,1.867394,2.013934,1.953053,1.950898,-0.336141,NaN,NaN,NaN,NaN,NaN,NaN,1.60,3.50,5.50,2.624982,3.753932,4.749475,1.65,3.40,5.00,1.650000,3.500000,5.500000,1.847198,1.837633,1.89003,1.914697,-0.323485,1.70,3.40,4.30,1.620000,3.250000,5.000000,1.931357,1.914883,-0.349454,2.958986,4.266214,4.992703,1.856226,2.157956,1.967555,1.955565,2.31698,3.468319,4.037422,1.620000,3.400000,6.000000,1.570000,3.500000,5.500000,NaN,NaN,NaN,1.57,3.30,5.50,46.0,1.730000,1.630000,3.700000,3.460000,6.00000,5.350000,39.0000,2.200000,2.070000,1.760000,1.690000,26.00000,-0.750000,2.000000,1.94000,1.960000,1.930000,3.1928,4.493388,5.08369,3.003213,4.243009,4.610582,1.890303,2.207561,1.821665,2.115279,1.995425,1.990041,1.940827,1.934651,-0.260914,2006-07,2006,8,5
3,2006-08-19,Newcastle,Wigan,2.0,1.0,H,1.0,0.0,H,M Atkinson,9.0,14.0,8.0,9.0,4.0,11.0,17.0,20.0,1.0,2.0,0.0,0.0,NaN,NaN,NaN,1.72,3.50,4.75,1.867394,2.013934,1.953053,1.950898,-0.336141,NaN,NaN,NaN,NaN,NaN,NaN,1.65,3.50,5.00,2.624982,3.753932,4.749475,1.70,3.35,4.70,1.700000,3.500000,5.000000,1.847198,1.837633,1.89003,1.914697,-0.323485,1.70,3.40,4.30,1.670000,3.200000,5.000000,1.931357,1.914883,-0.349454,2.958986,4.266214,4.992703,1.856226,2.157956,1.967555,1.955565,2.31698,3.468319,4.037422,1.670000,3.400000,5.250000,1.620000,3.500000,5.000000,NaN,NaN,NaN,1.72,3.20,4.33,44.0,1.870000,1.710000,3.550000,3.390000,5.25000,4.720000,39.0000,2.250000,2.120000,1.750000,1.660000,26.000

In [14]:
df['TotalGoals'] = df['FTHG'] + df['FTAG']
df['GoalsOver2_5'] = (df['TotalGoals'] > 2.5).astype(int)
df['BTTS'] = ((df['FTHG'] > 0) & (df['FTAG'] > 0)).astype(int)
df['Home_2plus'] = (df['FTHG'] >= 2).astype(int)
df['Away_2plus'] = (df['FTAG'] >= 2).astype(int)

In [15]:
df = df.sort_values('Date')
df['HomeTeam_mean_FTHG'] = (
    df.groupby('HomeTeam')['FTHG'].transform(lambda x: x.shift(1).expanding().mean())
)
df['AwayTeam_mean_FTAG'] = (
    df.groupby('AwayTeam')['FTAG'].transform(lambda x: x.shift(1).expanding().mean())
)

In [16]:
df['HomeTeam_str'] = df['HomeTeam']
df['AwayTeam_str'] = df['AwayTeam']
df = pd.get_dummies(df, columns=['HomeTeam', 'AwayTeam'])
df = df.rename(columns={'HomeTeam_str': 'HomeTeam', 'AwayTeam_str': 'AwayTeam'})

In [17]:
df

,Date,FTHG,FTAG,FTR,HTHG,HTAG,HTR,Referee,HS,AS,HST,AST,HC,AC,HF,AF,HY,AY,HR,AR,1XBH,1XBD,1XBA,B365H,B365D,B365A,B365>2.5,B365<2.5,B365AHH,B365AHA,B365AH,BFH,BFD,BFA,BFEH,BFED,BFEA,VCH,VCD,VCA,BSH,BSD,BSA,BWH,BWD,BWA,GBH,GBD,GBA,GB>2.5,GB<2.5,GBAHH,GBAHA,GBAH,IWH,IWD,IWA,LBH,LBD,LBA,LBAHH,LBAHA,LBAH,PSH,PSD,PSA,P>2.5,P<2.5,PAHH,PAHA,SOH,SOD,SOA,SBH,SBD,SBA,SJH,SJD,SJA,SYH,SYD,SYA,WHH,WHD,WHA,Bb1X2,BbMxH,BbAvH,BbMxD,BbAvD,BbMxA,BbAvA,BbOU,BbMx>2.5,BbAv>2.5,BbMx<2.5,BbAv<2.5,BbAH,BbAHh,BbMxAHH,BbAvAHH,BbMxAHA,BbAvAHA,MaxH,MaxD,MaxA,AvgH,AvgD,AvgA,Max>2.5,Max<2.5,Avg>2.5,Avg<2.5,MaxAHH,MaxAHA,AvgAHH,AvgAHA,AHh,Season,Year,Month,DayOfWeek,TotalGoals,GoalsOver2_5,BTTS,Home_2plus,Away_2plus,HomeTeam_mean_FTHG,AwayTeam_mean_FTAG,HomeTeam,AwayTeam,HomeTeam_Arsenal,HomeTeam_Aston Villa,HomeTeam_Birmingham,HomeTeam_Blackburn,HomeTeam_Blackpool,HomeTeam_Bolton,HomeTeam_Bournemouth,HomeTeam_Brentford,HomeTeam_Brighton,HomeTeam_Burnley,HomeTeam_Cardiff,HomeTeam_Charlton,HomeTeam_Chelsea,HomeTeam_Crystal Palace,HomeTeam_Derby,HomeTeam_Everton,HomeTeam_Fulham,HomeTeam_Huddersfield,HomeTeam_Hull,HomeTeam_Leeds,HomeTeam_Leicester,HomeTeam_Liverpool,HomeTeam_Luton,HomeTeam_Man City,HomeTeam_Man United,HomeTeam_Middlesbrough,HomeTeam_Newcastle,HomeTeam_Norwich,HomeTeam_Nott'm Forest,HomeTeam_Portsmouth,HomeTeam_QPR,HomeTeam_Reading,HomeTeam_Sheffield United,HomeTeam_Southampton,HomeTeam_Stoke,HomeTeam_Sunderland,HomeTeam_Swansea,HomeTeam_Tottenham,HomeTeam_Watford,HomeTeam_West Brom,HomeTeam_West Ham,HomeTeam_Wigan,HomeTeam_Wolves,AwayTeam_Arsenal,AwayTeam_Aston Villa,AwayTeam_Birmingham,AwayTeam_Blackburn,AwayTeam_Blackpool,AwayTeam_Bolton,AwayTeam_Bournemouth,AwayTeam_Brentford,AwayTeam_Brighton,AwayTeam_Burnley,AwayTeam_Cardiff,AwayTeam_Charlton,AwayTeam_Chelsea,AwayTeam_Crystal Palace,AwayTeam_Derby,AwayTeam_Everton,AwayTeam_Fulham,AwayTeam_Huddersfield,AwayTeam_Hull,AwayTeam_Leeds,AwayTeam_Leicester,AwayTeam_Liverpool,AwayTeam_Luton,AwayTeam_Man City,AwayTeam_Man United,AwayTeam_Middlesbrough,AwayTeam_Newcastle,AwayTeam_Norwich,AwayTeam_Nott'm Forest,AwayTeam_Portsmouth,AwayTeam_QPR,AwayTeam_Reading,AwayTeam_Sheffield United,AwayTeam_Southampton,AwayTeam_Stoke,AwayTeam_Sunderland,AwayTeam_Swansea,AwayTeam_Tottenham,AwayTeam_Watford,AwayTeam_West Brom,AwayTeam_West Ham,AwayTeam_Wigan,AwayTeam_Wolves
382,2002-08-17,2.0,2.0,D,1.0,0.0,H,N Barry,13.0,10.0,9.0,5.0,10.0,5.0,18.0,4.0,1.0,1.0,0.0,0.0,NaN,NaN,NaN,2.250,3.25,2.750,1.867394,2.013934,1.953053,1.950898,-0.336141,NaN,NaN,NaN,NaN,NaN,NaN,2.850784,4.027811,4.935736,2.624982,3.753932,4.749475,2.732814,3.917273,4.548528,2.390000,3.110000,2.810000,1.920000,1.720000,1.89003,1.914697,-0.323485,2.30,3.00,2.70,2.250000,3.200000,2.750000,1.931357,1.914883,-0.349454,2.958986,4.266214,4.992703,1.856226,2.157956,1.967555,1.955565,2.38000,3.200000,2.630000,2.400000,3.100000,2.800000,2.619415,3.778735,4.686697,NaN,NaN,NaN,2.30,3.1,2.75,NaN,2.940935,2.722174,4.173095,3.880495,5.45658,4.757603,36.5166,2.003017,1.902307,2.033331,1.930361,23.13209,-0.303451,2.004782,1.93259,2.173407,2.071169,3.1928,4.493388,5.08369,3.003213,4.243009,4.610582,1.890303,2.207561,1.821665,2.115279,1.995425,1.990041,1.940827,1.934651,-0.260914,2002-03,2002,8,5,4.0,1,1,1,1,NaN,NaN,Everton,Tottenham,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,True,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,True,False,False,False,False,False
381,2002-08-17,2.0,3.0,A,2.0,1.0,H,G Barber,5.0,21.0,5.0,12.0,3.0,6.0,10.0,12.0,0.0,3.0,1.0,0.0,NaN,NaN,NaN,2.800,3.25,2.200,1.867394,2.013934,1.953053,1.950898,-0.336141,NaN,NaN,NaN,NaN,NaN,NaN,2.850784,4.027811,4.935736,2.624982,

In [18]:
def add_recent_form_features(df, n_matches=5):
    base = df.copy()
    base = base.sort_values('Date')
    home_df = base[['Date', 'HomeTeam', 'FTHG', 'FTAG']].rename(
        columns={'HomeTeam': 'Team', 'FTHG': 'GoalsFor', 'FTAG': 'GoalsAgainst'})
    away_df = base[['Date', 'AwayTeam', 'FTAG', 'FTHG']].rename(
        columns={'AwayTeam': 'Team', 'FTAG': 'GoalsFor', 'FTHG': 'GoalsAgainst'})
    results = pd.concat([home_df, away_df], ignore_index=True)
    results = results.sort_values(['Team', 'Date'])
   
    def get_points(row):
        return 3 if row['GoalsFor'] > row['GoalsAgainst'] else (1 if row['GoalsFor'] == row['GoalsAgainst'] else 0)
    results['Points'] = results.apply(get_points, axis=1)
    results['RollingGF'] = results.groupby('Team')['GoalsFor'].transform(lambda x: x.shift(1).rolling(n_matches, min_periods=1).mean())
    results['RollingGA'] = results.groupby('Team')['GoalsAgainst'].transform(lambda x: x.shift(1).rolling(n_matches, min_periods=1).mean())
    results['RollingPoints'] = results.groupby('Team')['Points'].transform(lambda x: x.shift(1).rolling(n_matches, min_periods=1).sum())
  
    def get_form(row, team_col):
        team = row[team_col]
        date = row['Date']
        row_form = results[(results['Team'] == team) & (results['Date'] < date)].sort_values('Date').tail(1)
        if row_form.empty:
            return pd.Series([np.nan, np.nan, np.nan])
        return row_form[['RollingGF', 'RollingGA', 'RollingPoints']].values[0]
    base[['HomeRecentGF', 'HomeRecentGA', 'HomeRecentPts']] = base.apply(
        lambda row: get_form(row, 'HomeTeam'), axis=1, result_type='expand')
    base[['AwayRecentGF', 'AwayRecentGA', 'AwayRecentPts']] = base.apply(
        lambda row: get_form(row, 'AwayTeam'), axis=1, result_type='expand')
    return base

df = add_recent_form_features(df, n_matches=5)
df = df.dropna(subset=['HomeRecentGF', 'AwayRecentGF'])

In [19]:
df

,Date,FTHG,FTAG,FTR,HTHG,HTAG,HTR,Referee,HS,AS,HST,AST,HC,AC,HF,AF,HY,AY,HR,AR,1XBH,1XBD,1XBA,B365H,B365D,B365A,B365>2.5,B365<2.5,B365AHH,B365AHA,B365AH,BFH,BFD,BFA,BFEH,BFED,BFEA,VCH,VCD,VCA,BSH,BSD,BSA,BWH,BWD,BWA,GBH,GBD,GBA,GB>2.5,GB<2.5,GBAHH,GBAHA,GBAH,IWH,IWD,IWA,LBH,LBD,LBA,LBAHH,LBAHA,LBAH,PSH,PSD,PSA,P>2.5,P<2.5,PAHH,PAHA,SOH,SOD,SOA,SBH,SBD,SBA,SJH,SJD,SJA,SYH,SYD,SYA,WHH,WHD,WHA,Bb1X2,BbMxH,BbAvH,BbMxD,BbAvD,BbMxA,BbAvA,BbOU,BbMx>2.5,BbAv>2.5,BbMx<2.5,BbAv<2.5,BbAH,BbAHh,BbMxAHH,BbAvAHH,BbMxAHA,BbAvAHA,MaxH,MaxD,MaxA,AvgH,AvgD,AvgA,Max>2.5,Max<2.5,Avg>2.5,Avg<2.5,MaxAHH,MaxAHA,AvgAHH,AvgAHA,AHh,Season,Year,Month,DayOfWeek,TotalGoals,GoalsOver2_5,BTTS,Home_2plus,Away_2plus,HomeTeam_mean_FTHG,AwayTeam_mean_FTAG,HomeTeam,AwayTeam,HomeTeam_Arsenal,HomeTeam_Aston Villa,HomeTeam_Birmingham,HomeTeam_Blackburn,HomeTeam_Blackpool,HomeTeam_Bolton,HomeTeam_Bournemouth,HomeTeam_Brentford,HomeTeam_Brighton,HomeTeam_Burnley,HomeTeam_Cardiff,HomeTeam_Charlton,HomeTeam_Chelsea,HomeTeam_Crystal Palace,HomeTeam_Derby,HomeTeam_Everton,HomeTeam_Fulham,HomeTeam_Huddersfield,HomeTeam_Hull,HomeTeam_Leeds,HomeTeam_Leicester,HomeTeam_Liverpool,HomeTeam_Luton,HomeTeam_Man City,HomeTeam_Man United,HomeTeam_Middlesbrough,HomeTeam_Newcastle,HomeTeam_Norwich,HomeTeam_Nott'm Forest,HomeTeam_Portsmouth,HomeTeam_QPR,HomeTeam_Reading,HomeTeam_Sheffield United,HomeTeam_Southampton,HomeTeam_Stoke,HomeTeam_Sunderland,HomeTeam_Swansea,HomeTeam_Tottenham,HomeTeam_Watford,HomeTeam_West Brom,HomeTeam_West Ham,HomeTeam_Wigan,HomeTeam_Wolves,AwayTeam_Arsenal,AwayTeam_Aston Villa,AwayTeam_Birmingham,AwayTeam_Blackburn,AwayTeam_Blackpool,AwayTeam_Bolton,AwayTeam_Bournemouth,AwayTeam_Brentford,AwayTeam_Brighton,AwayTeam_Burnley,AwayTeam_Cardiff,AwayTeam_Charlton,AwayTeam_Chelsea,AwayTeam_Crystal Palace,AwayTeam_Derby,AwayTeam_Everton,AwayTeam_Fulham,AwayTeam_Huddersfield,AwayTeam_Hull,AwayTeam_Leeds,AwayTeam_Leicester,AwayTeam_Liverpool,AwayTeam_Luton,AwayTeam_Man City,AwayTeam_Man United,AwayTeam_Middlesbrough,AwayTeam_Newcastle,AwayTeam_Norwich,AwayTeam_Nott'm Forest,AwayTeam_Portsmouth,AwayTeam_QPR,AwayTeam_Reading,AwayTeam_Sheffield United,AwayTeam_Southampton,AwayTeam_Stoke,AwayTeam_Sunderland,AwayTeam_Swansea,AwayTeam_Tottenham,AwayTeam_Watford,AwayTeam_West Brom,AwayTeam_West Ham,AwayTeam_Wigan,AwayTeam_Wolves,HomeRecentGF,HomeRecentGA,HomeRecentPts,AwayRecentGF,AwayRecentGA,AwayRecentPts
400,2002-08-27,5.0,2.0,H,3.0,0.0,H,P Durkin,10.0,8.0,8.0,6.0,1.0,1.0,9.0,9.0,3.0,2.0,0.0,0.0,NaN,NaN,NaN,1.17,5.50,13.00,1.867394,2.013934,1.953053,1.950898,-0.336141,NaN,NaN,NaN,NaN,NaN,NaN,2.850784,4.027811,4.935736,2.624982,3.753932,4.749475,2.732814,3.917273,4.548528,1.150000,6.500000,12.000000,1.847198,1.837633,1.89003,1.914697,-0.323485,1.20,5.00,10.00,1.200000,5.000000,11.000000,1.931357,1.914883,-0.349454,2.958986,4.266214,4.992703,1.856226,2.157956,1.967555,1.955565,1.18000,6.000000,13.000000,1.170000,5.500000,13.000000,2.619415,3.778735,4.686697,NaN,NaN,NaN,1.16,5.50,12.00,NaN,2.940935,2.722174,4.173095,3.880495,5.45658,4.757603,36.5166,2.003017,1.902307,2.033331,1.930361,23.13209,-0.303451,2.004782,1.93259,2.173407,2.071169,3.1928,4.493388,5.08369,3.003213,4.243009,4.610582,1.890303,2.207561,1.821665,2.115279,1.995425,1.990041,1.940827,1.934651,-0.260914,2002-03,2002,8,1,7.0,1,1,1,1,2.000000,0.000000,Arsenal,West Brom,True,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,True,False,False,False,2.0,0.0,3.0,0.0,1.0,0.0
401,2002-08-27,0.0,1.0,A,0.0,1.0,A,A Wiley,10.0,7.0,5.0,1.0,6.0,9.0,9.0,23.0,1.0,2.0,0.0,0.0,NaN,NaN,NaN,2.40,3.2

In [20]:
df = df.sort_values('Date')
h2h_home_wins = []
team_stats = {}
home_positions = []
away_positions = []
for idx, row in df.iterrows():
    home = row['HomeTeam']
    away = row['AwayTeam']
    match_date = row['Date']
    prev_matches = df[
        (((df['HomeTeam'] == home) & (df['AwayTeam'] == away)) |
         ((df['HomeTeam'] == away) & (df['AwayTeam'] == home)))
        & (df['Date'] < match_date)
    ].sort_values('Date', ascending=False).head(5)
    home_wins = ((prev_matches['HomeTeam'] == home) & (prev_matches['FTHG'] > prev_matches['FTAG'])).sum()
    h2h_home_wins.append(home_wins)

    league_table = []
    for team, stats in team_stats.items():
        league_table.append({
            'team': team,
            'points': stats['points'],
            'gd': stats['gd'],
            'scored': stats['scored']
        })
    table_df = pd.DataFrame(league_table)
    if not table_df.empty:
        table_df = table_df.sort_values(['points', 'gd', 'scored'], ascending=[False, False, False])
        table_df['position'] = range(1, len(table_df) + 1)
        home_pos = table_df[table_df['team'] == home]['position'].values[0] if home in table_df['team'].values else len(table_df) + 1
        away_pos = table_df[table_df['team'] == away]['position'].values[0] if away in table_df['team'].values else len(table_df) + 1
    else:
        home_pos = away_pos = 1
    home_positions.append(home_pos)
    away_positions.append(away_pos)
  
    home_goals = row['FTHG']
    away_goals = row['FTAG']
    for team in [home, away]:
        if team not in team_stats:
            team_stats[team] = {'points': 0, 'gd': 0, 'scored': 0}
    if home_goals > away_goals:
        team_stats[home]['points'] += 3
    elif home_goals < away_goals:
        team_stats[away]['points'] += 3
    else:
        team_stats[home]['points'] += 1
        team_stats[away]['points'] += 1
    team_stats[home]['gd'] += home_goals - away_goals
    team_stats[away]['gd'] += away_goals - home_goals
    team_stats[home]['scored'] += home_goals
    team_stats[away]['scored'] += away_goals

df['h2h_home_wins_last5'] = h2h_home_wins
df['home_league_position'] = home_positions
df['away_league_position'] = away_positions
df['position_diff'] = df['home_league_position'] - df['away_league_position']

In [21]:
df

,Date,FTHG,FTAG,FTR,HTHG,HTAG,HTR,Referee,HS,AS,HST,AST,HC,AC,HF,AF,HY,AY,HR,AR,1XBH,1XBD,1XBA,B365H,B365D,B365A,B365>2.5,B365<2.5,B365AHH,B365AHA,B365AH,BFH,BFD,BFA,BFEH,BFED,BFEA,VCH,VCD,VCA,BSH,BSD,BSA,BWH,BWD,BWA,GBH,GBD,GBA,GB>2.5,GB<2.5,GBAHH,GBAHA,GBAH,IWH,IWD,IWA,LBH,LBD,LBA,LBAHH,LBAHA,LBAH,PSH,PSD,PSA,P>2.5,P<2.5,PAHH,PAHA,SOH,SOD,SOA,SBH,SBD,SBA,SJH,SJD,SJA,SYH,SYD,SYA,WHH,WHD,WHA,Bb1X2,BbMxH,BbAvH,BbMxD,BbAvD,BbMxA,BbAvA,BbOU,BbMx>2.5,BbAv>2.5,BbMx<2.5,BbAv<2.5,BbAH,BbAHh,BbMxAHH,BbAvAHH,BbMxAHA,BbAvAHA,MaxH,MaxD,MaxA,AvgH,AvgD,AvgA,Max>2.5,Max<2.5,Avg>2.5,Avg<2.5,MaxAHH,MaxAHA,AvgAHH,AvgAHA,AHh,Season,Year,Month,DayOfWeek,TotalGoals,GoalsOver2_5,BTTS,Home_2plus,Away_2plus,HomeTeam_mean_FTHG,AwayTeam_mean_FTAG,HomeTeam,AwayTeam,HomeTeam_Arsenal,HomeTeam_Aston Villa,HomeTeam_Birmingham,HomeTeam_Blackburn,HomeTeam_Blackpool,HomeTeam_Bolton,HomeTeam_Bournemouth,HomeTeam_Brentford,HomeTeam_Brighton,HomeTeam_Burnley,HomeTeam_Cardiff,HomeTeam_Charlton,HomeTeam_Chelsea,HomeTeam_Crystal Palace,HomeTeam_Derby,HomeTeam_Everton,HomeTeam_Fulham,HomeTeam_Huddersfield,HomeTeam_Hull,HomeTeam_Leeds,HomeTeam_Leicester,HomeTeam_Liverpool,HomeTeam_Luton,HomeTeam_Man City,HomeTeam_Man United,HomeTeam_Middlesbrough,HomeTeam_Newcastle,HomeTeam_Norwich,HomeTeam_Nott'm Forest,HomeTeam_Portsmouth,HomeTeam_QPR,HomeTeam_Reading,HomeTeam_Sheffield United,HomeTeam_Southampton,HomeTeam_Stoke,HomeTeam_Sunderland,HomeTeam_Swansea,HomeTeam_Tottenham,HomeTeam_Watford,HomeTeam_West Brom,HomeTeam_West Ham,HomeTeam_Wigan,HomeTeam_Wolves,AwayTeam_Arsenal,AwayTeam_Aston Villa,AwayTeam_Birmingham,AwayTeam_Blackburn,AwayTeam_Blackpool,AwayTeam_Bolton,AwayTeam_Bournemouth,AwayTeam_Brentford,AwayTeam_Brighton,AwayTeam_Burnley,AwayTeam_Cardiff,AwayTeam_Charlton,AwayTeam_Chelsea,AwayTeam_Crystal Palace,AwayTeam_Derby,AwayTeam_Everton,AwayTeam_Fulham,AwayTeam_Huddersfield,AwayTeam_Hull,AwayTeam_Leeds,AwayTeam_Leicester,AwayTeam_Liverpool,AwayTeam_Luton,AwayTeam_Man City,AwayTeam_Man United,AwayTeam_Middlesbrough,AwayTeam_Newcastle,AwayTeam_Norwich,AwayTeam_Nott'm Forest,AwayTeam_Portsmouth,AwayTeam_QPR,AwayTeam_Reading,AwayTeam_Sheffield United,AwayTeam_Southampton,AwayTeam_Stoke,AwayTeam_Sunderland,AwayTeam_Swansea,AwayTeam_Tottenham,AwayTeam_Watford,AwayTeam_West Brom,AwayTeam_West Ham,AwayTeam_Wigan,AwayTeam_Wolves,HomeRecentGF,HomeRecentGA,HomeRecentPts,AwayRecentGF,AwayRecentGA,AwayRecentPts,h2h_home_wins_last5,home_league_position,away_league_position,position_diff
400,2002-08-27,5.0,2.0,H,3.0,0.0,H,P Durkin,10.0,8.0,8.0,6.0,1.0,1.0,9.0,9.0,3.0,2.0,0.0,0.0,NaN,NaN,NaN,1.17,5.50,13.00,1.867394,2.013934,1.953053,1.950898,-0.336141,NaN,NaN,NaN,NaN,NaN,NaN,2.850784,4.027811,4.935736,2.624982,3.753932,4.749475,2.732814,3.917273,4.548528,1.150000,6.500000,12.000000,1.847198,1.837633,1.89003,1.914697,-0.323485,1.20,5.00,10.00,1.200000,5.000000,11.000000,1.931357,1.914883,-0.349454,2.958986,4.266214,4.992703,1.856226,2.157956,1.967555,1.955565,1.18000,6.000000,13.000000,1.170000,5.500000,13.000000,2.619415,3.778735,4.686697,NaN,NaN,NaN,1.16,5.50,12.00,NaN,2.940935,2.722174,4.173095,3.880495,5.45658,4.757603,36.5166,2.003017,1.902307,2.033331,1.930361,23.13209,-0.303451,2.004782,1.93259,2.173407,2.071169,3.1928,4.493388,5.08369,3.003213,4.243009,4.610582,1.890303,2.207561,1.821665,2.115279,1.995425,1.990041,1.940827,1.934651,-0.260914,2002-03,2002,8,1,7.0,1,1,1,1,2.000000,0.000000,Arsenal,West Brom,True,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,True,False,False,False,2.0,0.0,3.0,0.0,1.0,0.0,0,1,1,0
401,2002-08-27,0.0,1.0,A,0.0,

In [22]:
N = 5
df['HomePts'] = np.where(df['FTHG'] > df['FTAG'], 3, np.where(df['FTHG'] == df['FTAG'], 1, 0))
df['AwayPts'] = np.where(df['FTAG'] > df['FTHG'], 3, np.where(df['FTAG'] == df['FTHG'], 1, 0))
df['HomeRecentPts'] = (
    df.groupby('HomeTeam')['HomePts'].transform(lambda x: x.shift(1).rolling(N, min_periods=1).sum())
)
df['AwayRecentPts'] = (
    df.groupby('AwayTeam')['AwayPts'].transform(lambda x: x.shift(1).rolling(N, min_periods=1).sum())
)
df['HomeGoalDiff'] = df['FTHG'] - df['FTAG']
df['AwayGoalDiff'] = df['FTAG'] - df['FTHG']
df['HomeRecentGoalDiff'] = (
    df.groupby('HomeTeam')['HomeGoalDiff'].transform(lambda x: x.shift(1).rolling(N, min_periods=1).sum())
)
df['AwayRecentGoalDiff'] = (
    df.groupby('AwayTeam')['AwayGoalDiff'].transform(lambda x: x.shift(1).rolling(N, min_periods=1).sum())
)
df['RecentGoalDiff'] = df['HomeRecentGoalDiff'] - df['AwayRecentGoalDiff']
df['HomeRecentShotsOnTarget'] = (
    df.groupby('HomeTeam')['HST'].transform(lambda x: x.shift(1).rolling(N, min_periods=1).sum())
)
df['AwayRecentShotsOnTarget'] = (
    df.groupby('AwayTeam')['AST'].transform(lambda x: x.shift(1).rolling(N, min_periods=1).sum())
)
df['RecentShotsOnTargetDiff'] = df['HomeRecentShotsOnTarget'] - df['AwayRecentShotsOnTarget']
df['PosDiff'] = df['home_league_position'] - df['away_league_position']

for col in ['B365>2.5', 'B365<2.5']:
    median = df[col].median()
    df[f'{col}_missing'] = df[col].isna().astype(int)
    df[col] = df[col].fillna(median)

df['B365>2.5_implied_prob'] = 1 / df['B365>2.5']
df['B365<2.5_implied_prob'] = 1 / df['B365<2.5']
df['OddsMargin'] = df['B365H'] / df['B365A']
df['OU_OddsMargin'] = df['B365>2.5'] / df['B365<2.5']
df['OverUnderRatio'] = df['B365>2.5'] / df['B365<2.5']

df['Weekend'] = df['DayOfWeek'].isin([5, 6]).astype(int)
df['EarlySeason'] = (df['Month'] <= 3).astype(int)

In [23]:
df

,Date,FTHG,FTAG,FTR,HTHG,HTAG,HTR,Referee,HS,AS,HST,AST,HC,AC,HF,AF,HY,AY,HR,AR,1XBH,1XBD,1XBA,B365H,B365D,B365A,B365>2.5,B365<2.5,B365AHH,B365AHA,B365AH,BFH,BFD,BFA,BFEH,BFED,BFEA,VCH,VCD,VCA,BSH,BSD,BSA,BWH,BWD,BWA,GBH,GBD,GBA,GB>2.5,GB<2.5,GBAHH,GBAHA,GBAH,IWH,IWD,IWA,LBH,LBD,LBA,LBAHH,LBAHA,LBAH,PSH,PSD,PSA,P>2.5,P<2.5,PAHH,PAHA,SOH,SOD,SOA,SBH,SBD,SBA,SJH,SJD,SJA,SYH,SYD,SYA,WHH,WHD,WHA,Bb1X2,BbMxH,BbAvH,BbMxD,BbAvD,BbMxA,BbAvA,BbOU,BbMx>2.5,BbAv>2.5,BbMx<2.5,BbAv<2.5,BbAH,BbAHh,BbMxAHH,BbAvAHH,BbMxAHA,BbAvAHA,MaxH,MaxD,MaxA,AvgH,AvgD,AvgA,Max>2.5,Max<2.5,Avg>2.5,Avg<2.5,MaxAHH,MaxAHA,AvgAHH,AvgAHA,AHh,Season,Year,Month,DayOfWeek,TotalGoals,GoalsOver2_5,BTTS,Home_2plus,Away_2plus,HomeTeam_mean_FTHG,AwayTeam_mean_FTAG,HomeTeam,AwayTeam,HomeTeam_Arsenal,HomeTeam_Aston Villa,HomeTeam_Birmingham,HomeTeam_Blackburn,HomeTeam_Blackpool,HomeTeam_Bolton,HomeTeam_Bournemouth,HomeTeam_Brentford,HomeTeam_Brighton,HomeTeam_Burnley,HomeTeam_Cardiff,HomeTeam_Charlton,HomeTeam_Chelsea,HomeTeam_Crystal Palace,HomeTeam_Derby,HomeTeam_Everton,HomeTeam_Fulham,HomeTeam_Huddersfield,HomeTeam_Hull,HomeTeam_Leeds,HomeTeam_Leicester,HomeTeam_Liverpool,HomeTeam_Luton,HomeTeam_Man City,HomeTeam_Man United,HomeTeam_Middlesbrough,HomeTeam_Newcastle,HomeTeam_Norwich,HomeTeam_Nott'm Forest,HomeTeam_Portsmouth,HomeTeam_QPR,HomeTeam_Reading,HomeTeam_Sheffield United,HomeTeam_Southampton,HomeTeam_Stoke,HomeTeam_Sunderland,HomeTeam_Swansea,HomeTeam_Tottenham,HomeTeam_Watford,HomeTeam_West Brom,HomeTeam_West Ham,HomeTeam_Wigan,HomeTeam_Wolves,AwayTeam_Arsenal,AwayTeam_Aston Villa,AwayTeam_Birmingham,AwayTeam_Blackburn,AwayTeam_Blackpool,AwayTeam_Bolton,AwayTeam_Bournemouth,AwayTeam_Brentford,AwayTeam_Brighton,AwayTeam_Burnley,AwayTeam_Cardiff,AwayTeam_Charlton,AwayTeam_Chelsea,AwayTeam_Crystal Palace,AwayTeam_Derby,AwayTeam_Everton,AwayTeam_Fulham,AwayTeam_Huddersfield,AwayTeam_Hull,AwayTeam_Leeds,AwayTeam_Leicester,AwayTeam_Liverpool,AwayTeam_Luton,AwayTeam_Man City,AwayTeam_Man United,AwayTeam_Middlesbrough,AwayTeam_Newcastle,AwayTeam_Norwich,AwayTeam_Nott'm Forest,AwayTeam_Portsmouth,AwayTeam_QPR,AwayTeam_Reading,AwayTeam_Sheffield United,AwayTeam_Southampton,AwayTeam_Stoke,AwayTeam_Sunderland,AwayTeam_Swansea,AwayTeam_Tottenham,AwayTeam_Watford,AwayTeam_West Brom,AwayTeam_West Ham,AwayTeam_Wigan,AwayTeam_Wolves,HomeRecentGF,HomeRecentGA,HomeRecentPts,AwayRecentGF,AwayRecentGA,AwayRecentPts,h2h_home_wins_last5,home_league_position,away_league_position,position_diff,HomePts,AwayPts,HomeGoalDiff,AwayGoalDiff,HomeRecentGoalDiff,AwayRecentGoalDiff,RecentGoalDiff,HomeRecentShotsOnTarget,AwayRecentShotsOnTarget,RecentShotsOnTargetDiff,PosDiff,B365>2.5_missing,B365<2.5_missing,B365>2.5_implied_prob,B365<2.5_implied_prob,OddsMargin,OU_OddsMargin,OverUnderRatio,Weekend,EarlySeason
400,2002-08-27,5.0,2.0,H,3.0,0.0,H,P Durkin,10.0,8.0,8.0,6.0,1.0,1.0,9.0,9.0,3.0,2.0,0.0,0.0,NaN,NaN,NaN,1.17,5.50,13.00,1.867394,2.013934,1.953053,1.950898,-0.336141,NaN,NaN,NaN,NaN,NaN,NaN,2.850784,4.027811,4.935736,2.624982,3.753932,4.749475,2.732814,3.917273,4.548528,1.150000,6.500000,12.000000,1.847198,1.837633,1.89003,1.914697,-0.323485,1.20,5.00,10.00,1.200000,5.000000,11.000000,1.931357,1.914883,-0.349454,2.958986,4.266214,4.992703,1.856226,2.157956,1.967555,1.955565,1.18000,6.000000,13.000000,1.170000,5.500000,13.000000,2.619415,3.778735,4.686697,NaN,NaN,NaN,1.16,5.50,12.00,NaN,2.940935,2.722174,4.173095,3.880495,5.45658,4.757603,36.5166,2.003017,1.902307,2.033331,1.930361,23.13209,-0.303451,2.004782,1.93259,2.173407,2.071169,3.1928,4.493388,5.08369,3.003213,4.243009,4.610582,1.890303,2.207561,1.821665,2.115279,1.995425,1.990041,1.940827,1.934651,-0.260914,2002-03,2002,8,1,7.0,1,1,1,1,2.000000,0.000000,Arsenal,West Brom,True,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False

In [24]:
def rolling_ref_aggression(subdf):
    agg = subdf[['HY', 'AY', 'HR', 'AR']].shift(1).sum(axis=1)
    return agg.rolling(10, min_periods=1).mean()
df = df.sort_values('Date')
df['RefereeAggression'] = (
    df.groupby('Referee').apply(lambda subdf: rolling_ref_aggression(subdf)).reset_index(level=0, drop=True)
)

df = df.replace([np.inf, -np.inf], np.nan)
for col in df.select_dtypes(include=['number']):
    df[col] = df[col].fillna(df[col].median())
for col in df.select_dtypes(include=['object', 'category']):
    df[col] = df[col].fillna(df[col].mode()[0])

df = df.drop_duplicates().reset_index(drop=True)

/var/folders/tk/jl3yfcz52s9020z4_8dbfkvw0000gn/T/ipykernel_43954/1171296694.py:6: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  df.groupby('Referee').apply(lambda subdf: rolling_ref_aggression(subdf)).reset_index(level=0, drop=True)


In [25]:
df

,Date,FTHG,FTAG,FTR,HTHG,HTAG,HTR,Referee,HS,AS,HST,AST,HC,AC,HF,AF,HY,AY,HR,AR,1XBH,1XBD,1XBA,B365H,B365D,B365A,B365>2.5,B365<2.5,B365AHH,B365AHA,B365AH,BFH,BFD,BFA,BFEH,BFED,BFEA,VCH,VCD,VCA,BSH,BSD,BSA,BWH,BWD,BWA,GBH,GBD,GBA,GB>2.5,GB<2.5,GBAHH,GBAHA,GBAH,IWH,IWD,IWA,LBH,LBD,LBA,LBAHH,LBAHA,LBAH,PSH,PSD,PSA,P>2.5,P<2.5,PAHH,PAHA,SOH,SOD,SOA,SBH,SBD,SBA,SJH,SJD,SJA,SYH,SYD,SYA,WHH,WHD,WHA,Bb1X2,BbMxH,BbAvH,BbMxD,BbAvD,BbMxA,BbAvA,BbOU,BbMx>2.5,BbAv>2.5,BbMx<2.5,BbAv<2.5,BbAH,BbAHh,BbMxAHH,BbAvAHH,BbMxAHA,BbAvAHA,MaxH,MaxD,MaxA,AvgH,AvgD,AvgA,Max>2.5,Max<2.5,Avg>2.5,Avg<2.5,MaxAHH,MaxAHA,AvgAHH,AvgAHA,AHh,Season,Year,Month,DayOfWeek,TotalGoals,GoalsOver2_5,BTTS,Home_2plus,Away_2plus,HomeTeam_mean_FTHG,AwayTeam_mean_FTAG,HomeTeam,AwayTeam,HomeTeam_Arsenal,HomeTeam_Aston Villa,HomeTeam_Birmingham,HomeTeam_Blackburn,HomeTeam_Blackpool,HomeTeam_Bolton,HomeTeam_Bournemouth,HomeTeam_Brentford,HomeTeam_Brighton,HomeTeam_Burnley,HomeTeam_Cardiff,HomeTeam_Charlton,HomeTeam_Chelsea,HomeTeam_Crystal Palace,HomeTeam_Derby,HomeTeam_Everton,HomeTeam_Fulham,HomeTeam_Huddersfield,HomeTeam_Hull,HomeTeam_Leeds,HomeTeam_Leicester,HomeTeam_Liverpool,HomeTeam_Luton,HomeTeam_Man City,HomeTeam_Man United,HomeTeam_Middlesbrough,HomeTeam_Newcastle,HomeTeam_Norwich,HomeTeam_Nott'm Forest,HomeTeam_Portsmouth,HomeTeam_QPR,HomeTeam_Reading,HomeTeam_Sheffield United,HomeTeam_Southampton,HomeTeam_Stoke,HomeTeam_Sunderland,HomeTeam_Swansea,HomeTeam_Tottenham,HomeTeam_Watford,HomeTeam_West Brom,HomeTeam_West Ham,HomeTeam_Wigan,HomeTeam_Wolves,AwayTeam_Arsenal,AwayTeam_Aston Villa,AwayTeam_Birmingham,AwayTeam_Blackburn,AwayTeam_Blackpool,AwayTeam_Bolton,AwayTeam_Bournemouth,AwayTeam_Brentford,AwayTeam_Brighton,AwayTeam_Burnley,AwayTeam_Cardiff,AwayTeam_Charlton,AwayTeam_Chelsea,AwayTeam_Crystal Palace,AwayTeam_Derby,AwayTeam_Everton,AwayTeam_Fulham,AwayTeam_Huddersfield,AwayTeam_Hull,AwayTeam_Leeds,AwayTeam_Leicester,AwayTeam_Liverpool,AwayTeam_Luton,AwayTeam_Man City,AwayTeam_Man United,AwayTeam_Middlesbrough,AwayTeam_Newcastle,AwayTeam_Norwich,AwayTeam_Nott'm Forest,AwayTeam_Portsmouth,AwayTeam_QPR,AwayTeam_Reading,AwayTeam_Sheffield United,AwayTeam_Southampton,AwayTeam_Stoke,AwayTeam_Sunderland,AwayTeam_Swansea,AwayTeam_Tottenham,AwayTeam_Watford,AwayTeam_West Brom,AwayTeam_West Ham,AwayTeam_Wigan,AwayTeam_Wolves,HomeRecentGF,HomeRecentGA,HomeRecentPts,AwayRecentGF,AwayRecentGA,AwayRecentPts,h2h_home_wins_last5,home_league_position,away_league_position,position_diff,HomePts,AwayPts,HomeGoalDiff,AwayGoalDiff,HomeRecentGoalDiff,AwayRecentGoalDiff,RecentGoalDiff,HomeRecentShotsOnTarget,AwayRecentShotsOnTarget,RecentShotsOnTargetDiff,PosDiff,B365>2.5_missing,B365<2.5_missing,B365>2.5_implied_prob,B365<2.5_implied_prob,OddsMargin,OU_OddsMargin,OverUnderRatio,Weekend,EarlySeason,RefereeAggression
0,2002-08-27,5.0,2.0,H,3.0,0.0,H,P Durkin,10.0,8.0,8.0,6.0,1.0,1.0,9.0,9.0,3.0,2.0,0.0,0.0,NaN,NaN,NaN,1.17,5.50,13.00,1.867394,2.013934,1.953053,1.950898,-0.336141,NaN,NaN,NaN,NaN,NaN,NaN,2.850784,4.027811,4.935736,2.624982,3.753932,4.749475,2.732814,3.917273,4.548528,1.150000,6.500000,12.000000,1.847198,1.837633,1.89003,1.914697,-0.323485,1.20,5.00,10.00,1.200000,5.000000,11.000000,1.931357,1.914883,-0.349454,2.958986,4.266214,4.992703,1.856226,2.157956,1.967555,1.955565,1.18000,6.000000,13.000000,1.170000,5.500000,13.000000,2.619415,3.778735,4.686697,NaN,NaN,NaN,1.16,5.50,12.00,41.0,2.940935,2.722174,4.173095,3.880495,5.45658,4.757603,36.5166,2.003017,1.902307,2.033331,1.930361,23.13209,-0.303451,2.004782,1.93259,2.173407,2.071169,3.1928,4.493388,5.08369,3.003213,4.243009,4.610582,1.890303,2.207561,1.821665,2.115279,1.995425,1.990041,1.940827,1.934651,-0.260914,2002-03,2002,8,1,7.0,1,1,1,1,2.000000,0.000000,Arsenal,West Brom,True,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,

In [26]:
threshold = 0.95 
df = df.loc[:, df.isnull().mean() < threshold]

In [27]:
df

,Date,FTHG,FTAG,FTR,HTHG,HTAG,HTR,Referee,HS,AS,HST,AST,HC,AC,HF,AF,HY,AY,HR,AR,B365H,B365D,B365A,B365>2.5,B365<2.5,B365AHH,B365AHA,B365AH,VCH,VCD,VCA,BSH,BSD,BSA,BWH,BWD,BWA,GBH,GBD,GBA,GB>2.5,GB<2.5,GBAHH,GBAHA,GBAH,IWH,IWD,IWA,LBH,LBD,LBA,LBAHH,LBAHA,LBAH,PSH,PSD,PSA,P>2.5,P<2.5,PAHH,PAHA,SOH,SOD,SOA,SBH,SBD,SBA,SJH,SJD,SJA,WHH,WHD,WHA,Bb1X2,BbMxH,BbAvH,BbMxD,BbAvD,BbMxA,BbAvA,BbOU,BbMx>2.5,BbAv>2.5,BbMx<2.5,BbAv<2.5,BbAH,BbAHh,BbMxAHH,BbAvAHH,BbMxAHA,BbAvAHA,MaxH,MaxD,MaxA,AvgH,AvgD,AvgA,Max>2.5,Max<2.5,Avg>2.5,Avg<2.5,MaxAHH,MaxAHA,AvgAHH,AvgAHA,AHh,Season,Year,Month,DayOfWeek,TotalGoals,GoalsOver2_5,BTTS,Home_2plus,Away_2plus,HomeTeam_mean_FTHG,AwayTeam_mean_FTAG,HomeTeam,AwayTeam,HomeTeam_Arsenal,HomeTeam_Aston Villa,HomeTeam_Birmingham,HomeTeam_Blackburn,HomeTeam_Blackpool,HomeTeam_Bolton,HomeTeam_Bournemouth,HomeTeam_Brentford,HomeTeam_Brighton,HomeTeam_Burnley,HomeTeam_Cardiff,HomeTeam_Charlton,HomeTeam_Chelsea,HomeTeam_Crystal Palace,HomeTeam_Derby,HomeTeam_Everton,HomeTeam_Fulham,HomeTeam_Huddersfield,HomeTeam_Hull,HomeTeam_Leeds,HomeTeam_Leicester,HomeTeam_Liverpool,HomeTeam_Luton,HomeTeam_Man City,HomeTeam_Man United,HomeTeam_Middlesbrough,HomeTeam_Newcastle,HomeTeam_Norwich,HomeTeam_Nott'm Forest,HomeTeam_Portsmouth,HomeTeam_QPR,HomeTeam_Reading,HomeTeam_Sheffield United,HomeTeam_Southampton,HomeTeam_Stoke,HomeTeam_Sunderland,HomeTeam_Swansea,HomeTeam_Tottenham,HomeTeam_Watford,HomeTeam_West Brom,HomeTeam_West Ham,HomeTeam_Wigan,HomeTeam_Wolves,AwayTeam_Arsenal,AwayTeam_Aston Villa,AwayTeam_Birmingham,AwayTeam_Blackburn,AwayTeam_Blackpool,AwayTeam_Bolton,AwayTeam_Bournemouth,AwayTeam_Brentford,AwayTeam_Brighton,AwayTeam_Burnley,AwayTeam_Cardiff,AwayTeam_Charlton,AwayTeam_Chelsea,AwayTeam_Crystal Palace,AwayTeam_Derby,AwayTeam_Everton,AwayTeam_Fulham,AwayTeam_Huddersfield,AwayTeam_Hull,AwayTeam_Leeds,AwayTeam_Leicester,AwayTeam_Liverpool,AwayTeam_Luton,AwayTeam_Man City,AwayTeam_Man United,AwayTeam_Middlesbrough,AwayTeam_Newcastle,AwayTeam_Norwich,AwayTeam_Nott'm Forest,AwayTeam_Portsmouth,AwayTeam_QPR,AwayTeam_Reading,AwayTeam_Sheffield United,AwayTeam_Southampton,AwayTeam_Stoke,AwayTeam_Sunderland,AwayTeam_Swansea,AwayTeam_Tottenham,AwayTeam_Watford,AwayTeam_West Brom,AwayTeam_West Ham,AwayTeam_Wigan,AwayTeam_Wolves,HomeRecentGF,HomeRecentGA,HomeRecentPts,AwayRecentGF,AwayRecentGA,AwayRecentPts,h2h_home_wins_last5,home_league_position,away_league_position,position_diff,HomePts,AwayPts,HomeGoalDiff,AwayGoalDiff,HomeRecentGoalDiff,AwayRecentGoalDiff,RecentGoalDiff,HomeRecentShotsOnTarget,AwayRecentShotsOnTarget,RecentShotsOnTargetDiff,PosDiff,B365>2.5_missing,B365<2.5_missing,B365>2.5_implied_prob,B365<2.5_implied_prob,OddsMargin,OU_OddsMargin,OverUnderRatio,Weekend,EarlySeason,RefereeAggression
0,2002-08-27,5.0,2.0,H,3.0,0.0,H,P Durkin,10.0,8.0,8.0,6.0,1.0,1.0,9.0,9.0,3.0,2.0,0.0,0.0,1.17,5.50,13.00,1.867394,2.013934,1.953053,1.950898,-0.336141,2.850784,4.027811,4.935736,2.624982,3.753932,4.749475,2.732814,3.917273,4.548528,1.150000,6.500000,12.000000,1.847198,1.837633,1.89003,1.914697,-0.323485,1.20,5.00,10.00,1.200000,5.000000,11.000000,1.931357,1.914883,-0.349454,2.958986,4.266214,4.992703,1.856226,2.157956,1.967555,1.955565,1.18000,6.000000,13.000000,1.170000,5.500000,13.000000,2.619415,3.778735,4.686697,1.16,5.50,12.00,41.0,2.940935,2.722174,4.173095,3.880495,5.45658,4.757603,36.5166,2.003017,1.902307,2.033331,1.930361,23.13209,-0.303451,2.004782,1.93259,2.173407,2.071169,3.1928,4.493388,5.08369,3.003213,4.243009,4.610582,1.890303,2.207561,1.821665,2.115279,1.995425,1.990041,1.940827,1.934651,-0.260914,2002-03,2002,8,1,7.0,1,1,1,1,2.000000,0.000000,Arsenal,West Brom,True,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,False,

In [28]:
#pd.set_option('display.max_rows', None)
pd.reset_option('display.max_rows')
print("Shape of df:", df.shape)

missing_counts = df.isnull().sum().sort_values(ascending=False)
print("\nMissing values per column:")
print(missing_counts)

missing_perc = (df.isnull().mean() * 100).sort_values(ascending=False)
print("\nPercentage missing per column:")
print(missing_perc)

high_nan_cols = missing_perc[missing_perc > 95].index.tolist()
print("\nColumns with >95% missing values:", high_nan_cols)

print(f"\nTotal columns: {df.shape[1]}")
print(f"Columns with no missing: {(missing_counts==0).sum()}")
print(f"Columns with >50% missing: {(missing_perc > 50).sum()}")

Shape of df: (7901, 236)

Missing values per column:
Date                         0
AwayTeam_Arsenal             0
HomeTeam_Reading             0
HomeTeam_Sheffield United    0
HomeTeam_Southampton         0
                            ..
BbAv>2.5                     0
BbMx<2.5                     0
BbAv<2.5                     0
BbAH                         0
RefereeAggression            0
Length: 236, dtype: int64

Percentage missing per column:
Date                         0.0
AwayTeam_Arsenal             0.0
HomeTeam_Reading             0.0
HomeTeam_Sheffield United    0.0
HomeTeam_Southampton         0.0
                            ... 
BbAv>2.5                     0.0
BbMx<2.5                     0.0
BbAv<2.5                     0.0
BbAH                         0.0
RefereeAggression            0.0
Length: 236, dtype: float64

Columns with >95% missing values: []

Total columns: 236
Columns with no missing: 236
Columns with >50% missing: 0


In [29]:
print("All columns in df:")
print(df.columns.tolist())

All columns in df:
['Date', 'FTHG', 'FTAG', 'FTR', 'HTHG', 'HTAG', 'HTR', 'Referee', 'HS', 'AS', 'HST', 'AST', 'HC', 'AC', 'HF', 'AF', 'HY', 'AY', 'HR', 'AR', 'B365H', 'B365D', 'B365A', 'B365>2.5', 'B365<2.5', 'B365AHH', 'B365AHA', 'B365AH', 'VCH', 'VCD', 'VCA', 'BSH', 'BSD', 'BSA', 'BWH', 'BWD', 'BWA', 'GBH', 'GBD', 'GBA', 'GB>2.5', 'GB<2.5', 'GBAHH', 'GBAHA', 'GBAH', 'IWH', 'IWD', 'IWA', 'LBH', 'LBD', 'LBA', 'LBAHH', 'LBAHA', 'LBAH', 'PSH', 'PSD', 'PSA', 'P>2.5', 'P<2.5', 'PAHH', 'PAHA', 'SOH', 'SOD', 'SOA', 'SBH', 'SBD', 'SBA', 'SJH', 'SJD', 'SJA', 'WHH', 'WHD', 'WHA', 'Bb1X2', 'BbMxH', 'BbAvH', 'BbMxD', 'BbAvD', 'BbMxA', 'BbAvA', 'BbOU', 'BbMx>2.5', 'BbAv>2.5', 'BbMx<2.5', 'BbAv<2.5', 'BbAH', 'BbAHh', 'BbMxAHH', 'BbAvAHH', 'BbMxAHA', 'BbAvAHA', 'MaxH', 'MaxD', 'MaxA', 'AvgH', 'AvgD', 'AvgA', 'Max>2.5', 'Max<2.5', 'Avg>2.5', 'Avg<2.5', 'MaxAHH', 'MaxAHA', 'AvgAHH', 'AvgAHA', 'AHh', 'Season', 'Year', 'Month', 'DayOfWeek', 'TotalGoals', 'GoalsOver2_5', 'BTTS', 'Home_2plus', 'Away_2plu

In [30]:
odds_key = [
    "1XBH", "1XBD", "1XBA", "B365H", "B365D", "B365A", "BFH", "BFD", "BFA", "BFDH", "BFDD", "BFDA", "BMGMH", "BMGMD", "BMGMA",
    "BVH", "BVD", "BVA", "BSH", "BSD", "BSA", "BWH", "BWD", "BWA", "CLH", "CLD", "CLA", "GBH", "GBD", "GBA", "IWH", "IWD", "IWA",
    "LBH", "LBD", "LBA", "PSH", "PSD", "PSA", "PH", "PD", "PA", "SOH", "SOD", "SOA", "SBH", "SBD", "SBA", "SJH", "SJD", "SJA",
    "SYH", "SYD", "SYA", "VCH", "VCD", "VCA", "WHH", "WHD", "WHA", "Bb1X2", "BbMxH", "BbAvH", "BbMxD", "BbAvD", "BbMxA", "BbAvA",
    "MaxH", "MaxD", "MaxA", "AvgH", "AvgD", "AvgA", "BFEH", "BFED", "BFEA",
    "BbOU", "BbMx>2.5", "BbAv>2.5", "BbMx<2.5", "BbAv<2.5", "GB>2.5", "GB<2.5", "B365>2.5", "B365<2.5", "P>2.5", "P<2.5",
    "Max>2.5", "Max<2.5", "Avg>2.5", "Avg<2.5",
    "BbAH", "BbAHh", "AHh", "BbMxAHH", "BbAvAHH", "BbMxAHA", "BbAvAHA", "GBAHH", "GBAHA", "GBAH",
    "LBAHH", "LBAHA", "LBAH", "B365AHH", "B365AHA", "B365AH", "PAHH", "PAHA", "MaxAHH", "MaxAHA", "AvgAHH", "AvgAHA"
]

odds_in_df = [col for col in odds_key if col in df.columns]
print("Betting odds columns present in your DataFrame:")
print(odds_in_df)

odds_missing = [col for col in odds_key if col not in df.columns]
print("\nBetting odds columns missing from your DataFrame:")
print(odds_missing)

Betting odds columns present in your DataFrame:
['B365H', 'B365D', 'B365A', 'BSH', 'BSD', 'BSA', 'BWH', 'BWD', 'BWA', 'GBH', 'GBD', 'GBA', 'IWH', 'IWD', 'IWA', 'LBH', 'LBD', 'LBA', 'PSH', 'PSD', 'PSA', 'SOH', 'SOD', 'SOA', 'SBH', 'SBD', 'SBA', 'SJH', 'SJD', 'SJA', 'VCH', 'VCD', 'VCA', 'WHH', 'WHD', 'WHA', 'Bb1X2', 'BbMxH', 'BbAvH', 'BbMxD', 'BbAvD', 'BbMxA', 'BbAvA', 'MaxH', 'MaxD', 'MaxA', 'AvgH', 'AvgD', 'AvgA', 'BbOU', 'BbMx>2.5', 'BbAv>2.5', 'BbMx<2.5', 'BbAv<2.5', 'GB>2.5', 'GB<2.5', 'B365>2.5', 'B365<2.5', 'P>2.5', 'P<2.5', 'Max>2.5', 'Max<2.5', 'Avg>2.5', 'Avg<2.5', 'BbAH', 'BbAHh', 'AHh', 'BbMxAHH', 'BbAvAHH', 'BbMxAHA', 'BbAvAHA', 'GBAHH', 'GBAHA', 'GBAH', 'LBAHH', 'LBAHA', 'LBAH', 'B365AHH', 'B365AHA', 'B365AH', 'PAHH', 'PAHA', 'MaxAHH', 'MaxAHA', 'AvgAHH', 'AvgAHA']

Betting odds columns missing from your DataFrame:
['1XBH', '1XBD', '1XBA', 'BFH', 'BFD', 'BFA', 'BFDH', 'BFDD', 'BFDA', 'BMGMH', 'BMGMD', 'BMGMA', 'BVH', 'BVD', 'BVA', 'CLH', 'CLD', 'CLA', 'PH', 'PD', 'PA', 'SYH

In [31]:
df = df.copy()

if 'B365H_prob' not in df.columns:
    df.loc[:, 'B365H_prob'] = 1 / df['B365H']
if 'B365D_prob' not in df.columns:
    df.loc[:, 'B365D_prob'] = 1 / df['B365D']
if 'B365A_prob' not in df.columns:
    df.loc[:, 'B365A_prob'] = 1 / df['B365A']

if not all(col in df.columns for col in ['B365H_prob_norm', 'B365D_prob_norm', 'B365A_prob_norm']):
    prob_sum = df['B365H_prob'] + df['B365D_prob'] + df['B365A_prob']
    df.loc[:, 'B365H_prob_norm'] = df['B365H_prob'] / prob_sum
    df.loc[:, 'B365D_prob_norm'] = df['B365D_prob'] / prob_sum
    df.loc[:, 'B365A_prob_norm'] = df['B365A_prob'] / prob_sum

home_win_cols = ['B365H', 'BSH', 'BWH', 'GBH', 'IWH', 'LBH', 'PSH', 'SOH', 'SBH', 'SJH', 'VCH', 'WHH']
draw_cols = ['B365D', 'BSD', 'BWD', 'GBD', 'IWD', 'LBD', 'PSD', 'SOD', 'SBD', 'SJD', 'VCD', 'WHD']
away_win_cols = ['B365A', 'BSA', 'BWA', 'GBA', 'IWA', 'LBA', 'PSA', 'SOA', 'SBA', 'SJA', 'VCA', 'WHA']

for feat, cols in zip(['cons_mean_home', 'cons_mean_draw', 'cons_mean_away'], [home_win_cols, draw_cols, away_win_cols]):
    if feat not in df.columns:
        df.loc[:, feat] = df[cols].mean(axis=1)

for feat, cols in zip(['cons_min_home', 'cons_min_draw', 'cons_min_away'], [home_win_cols, draw_cols, away_win_cols]):
    if feat not in df.columns:
        df.loc[:, feat] = df[cols].min(axis=1)

for feat, cols in zip(['cons_max_home', 'cons_max_draw', 'cons_max_away'], [home_win_cols, draw_cols, away_win_cols]):
    if feat not in df.columns:
        df.loc[:, feat] = df[cols].max(axis=1)

if 'home_odds_spread' not in df.columns:
    df.loc[:, 'home_odds_spread'] = df['cons_max_home'] - df['cons_min_home']
if 'draw_odds_spread' not in df.columns:
    df.loc[:, 'draw_odds_spread'] = df['cons_max_draw'] - df['cons_min_draw']
if 'away_odds_spread' not in df.columns:
    df.loc[:, 'away_odds_spread'] = df['cons_max_away'] - df['cons_min_away']

if 'B365_overround' not in df.columns:
    df.loc[:, 'B365_overround'] = (1 / df['B365H']) + (1 / df['B365D']) + (1 / df['B365A'])
if 'cons_overround' not in df.columns:
    df.loc[:, 'cons_overround'] = (1 / df['cons_mean_home']) + (1 / df['cons_mean_draw']) + (1 / df['cons_mean_away'])

ou_cols_over = ['BbMx>2.5', 'BbAv>2.5', 'GB>2.5', 'B365>2.5', 'P>2.5', 'Max>2.5', 'Avg>2.5']
ou_cols_under = ['BbMx<2.5', 'BbAv<2.5', 'GB<2.5', 'B365<2.5', 'P<2.5', 'Max<2.5', 'Avg<2.5']

if 'cons_mean_over2.5' not in df.columns:
    df.loc[:, 'cons_mean_over2.5'] = df[ou_cols_over].mean(axis=1)
if 'cons_mean_under2.5' not in df.columns:
    df.loc[:, 'cons_mean_under2.5'] = df[ou_cols_under].mean(axis=1)
if 'cons_over2.5_prob' not in df.columns:
    df.loc[:, 'cons_over2.5_prob'] = 1 / df['cons_mean_over2.5']
if 'cons_under2.5_prob' not in df.columns:
    df.loc[:, 'cons_under2.5_prob'] = 1 / df['cons_mean_under2.5']

ah_home_cols = ['BbMxAHH', 'BbAvAHH', 'GBAHH', 'LBAHH', 'B365AHH', 'PAHH', 'MaxAHH', 'AvgAHH']
ah_away_cols = ['BbMxAHA', 'BbAvAHA', 'GBAHA', 'LBAHA', 'B365AHA', 'PAHA', 'MaxAHA', 'AvgAHA']
if all(col in df.columns for col in ah_home_cols) and 'mean_AH_home' not in df.columns:
    df.loc[:, 'mean_AH_home'] = df[ah_home_cols].mean(axis=1)
if all(col in df.columns for col in ah_away_cols) and 'mean_AH_away' not in df.columns:
    df.loc[:, 'mean_AH_away'] = df[ah_away_cols].mean(axis=1)
if 'AHh' in df.columns and 'market_AHh' not in df.columns:
    df.loc[:, 'market_AHh'] = df['AHh']

if 'target_home_plus_two' not in df.columns and 'FTHG' in df.columns:
    df.loc[:, 'target_home_plus_two'] = (df['FTHG'] >= 2).astype(int)
if 'target_away_plus_two' not in df.columns and 'FTAG' in df.columns:
    df.loc[:, 'target_away_plus_two'] = (df['FTAG'] >= 2).astype(int)

if 'home_away_odds_ratio' not in df.columns:
    df.loc[:, 'home_away_odds_ratio'] = df['cons_mean_home'] / df['cons_mean_away']

In [32]:
df

,Date,FTHG,FTAG,FTR,HTHG,HTAG,HTR,Referee,HS,AS,HST,AST,HC,AC,HF,AF,HY,AY,HR,AR,B365H,B365D,B365A,B365>2.5,B365<2.5,B365AHH,B365AHA,B365AH,VCH,VCD,VCA,BSH,BSD,BSA,BWH,BWD,BWA,GBH,GBD,GBA,GB>2.5,GB<2.5,GBAHH,GBAHA,GBAH,IWH,IWD,IWA,LBH,LBD,LBA,LBAHH,LBAHA,LBAH,PSH,PSD,PSA,P>2.5,P<2.5,PAHH,PAHA,SOH,SOD,SOA,SBH,SBD,SBA,SJH,SJD,SJA,WHH,WHD,WHA,Bb1X2,BbMxH,BbAvH,BbMxD,BbAvD,BbMxA,BbAvA,BbOU,BbMx>2.5,BbAv>2.5,BbMx<2.5,BbAv<2.5,BbAH,BbAHh,BbMxAHH,BbAvAHH,BbMxAHA,BbAvAHA,MaxH,MaxD,MaxA,AvgH,AvgD,AvgA,Max>2.5,Max<2.5,Avg>2.5,Avg<2.5,MaxAHH,MaxAHA,AvgAHH,AvgAHA,AHh,Season,Year,Month,DayOfWeek,TotalGoals,GoalsOver2_5,BTTS,Home_2plus,Away_2plus,HomeTeam_mean_FTHG,AwayTeam_mean_FTAG,HomeTeam,AwayTeam,HomeTeam_Arsenal,HomeTeam_Aston Villa,HomeTeam_Birmingham,HomeTeam_Blackburn,HomeTeam_Blackpool,HomeTeam_Bolton,HomeTeam_Bournemouth,HomeTeam_Brentford,HomeTeam_Brighton,HomeTeam_Burnley,HomeTeam_Cardiff,HomeTeam_Charlton,HomeTeam_Chelsea,HomeTeam_Crystal Palace,HomeTeam_Derby,HomeTeam_Everton,HomeTeam_Fulham,HomeTeam_Huddersfield,HomeTeam_Hull,HomeTeam_Leeds,HomeTeam_Leicester,HomeTeam_Liverpool,HomeTeam_Luton,HomeTeam_Man City,HomeTeam_Man United,HomeTeam_Middlesbrough,HomeTeam_Newcastle,HomeTeam_Norwich,HomeTeam_Nott'm Forest,HomeTeam_Portsmouth,HomeTeam_QPR,HomeTeam_Reading,HomeTeam_Sheffield United,HomeTeam_Southampton,HomeTeam_Stoke,HomeTeam_Sunderland,HomeTeam_Swansea,HomeTeam_Tottenham,HomeTeam_Watford,HomeTeam_West Brom,HomeTeam_West Ham,HomeTeam_Wigan,HomeTeam_Wolves,AwayTeam_Arsenal,AwayTeam_Aston Villa,AwayTeam_Birmingham,AwayTeam_Blackburn,AwayTeam_Blackpool,AwayTeam_Bolton,AwayTeam_Bournemouth,AwayTeam_Brentford,AwayTeam_Brighton,AwayTeam_Burnley,AwayTeam_Cardiff,AwayTeam_Charlton,AwayTeam_Chelsea,AwayTeam_Crystal Palace,AwayTeam_Derby,AwayTeam_Everton,AwayTeam_Fulham,AwayTeam_Huddersfield,AwayTeam_Hull,AwayTeam_Leeds,AwayTeam_Leicester,AwayTeam_Liverpool,AwayTeam_Luton,AwayTeam_Man City,AwayTeam_Man United,AwayTeam_Middlesbrough,AwayTeam_Newcastle,AwayTeam_Norwich,AwayTeam_Nott'm Forest,AwayTeam_Portsmouth,AwayTeam_QPR,AwayTeam_Reading,AwayTeam_Sheffield United,AwayTeam_Southampton,AwayTeam_Stoke,AwayTeam_Sunderland,AwayTeam_Swansea,AwayTeam_Tottenham,AwayTeam_Watford,AwayTeam_West Brom,AwayTeam_West Ham,AwayTeam_Wigan,AwayTeam_Wolves,HomeRecentGF,HomeRecentGA,HomeRecentPts,AwayRecentGF,AwayRecentGA,AwayRecentPts,h2h_home_wins_last5,home_league_position,away_league_position,position_diff,HomePts,AwayPts,HomeGoalDiff,AwayGoalDiff,HomeRecentGoalDiff,AwayRecentGoalDiff,RecentGoalDiff,HomeRecentShotsOnTarget,AwayRecentShotsOnTarget,RecentShotsOnTargetDiff,PosDiff,B365>2.5_missing,B365<2.5_missing,B365>2.5_implied_prob,B365<2.5_implied_prob,OddsMargin,OU_OddsMargin,OverUnderRatio,Weekend,EarlySeason,RefereeAggression,B365H_prob,B365D_prob,B365A_prob,B365H_prob_norm,B365D_prob_norm,B365A_prob_norm,cons_mean_home,cons_mean_draw,cons_mean_away,cons_min_home,cons_min_draw,cons_min_away,cons_max_home,cons_max_draw,cons_max_away,home_odds_spread,draw_odds_spread,away_odds_spread,B365_overround,cons_overround,cons_mean_over2.5,cons_mean_under2.5,cons_over2.5_prob,cons_under2.5_prob,mean_AH_home,mean_AH_away,market_AHh,target_home_plus_two,target_away_plus_two,home_away_odds_ratio
0,2002-08-27,5.0,2.0,H,3.0,0.0,H,P Durkin,10.0,8.0,8.0,6.0,1.0,1.0,9.0,9.0,3.0,2.0,0.0,0.0,1.17,5.50,13.00,1.867394,2.013934,1.953053,1.950898,-0.336141,2.850784,4.027811,4.935736,2.624982,3.753932,4.749475,2.732814,3.917273,4.548528,1.150000,6.500000,12.000000,1.847198,1.837633,1.89003,1.914697,-0.323485,1.20,5.00,10.00,1.200000,5.000000,11.000000,1.931357,1.914883,-0.349454,2.958986,4.266214,4.992703,1.856226,2.157956,1.967555,1.955565,1.18000,6.000000,13.000000,1.170000,5.500000,13.000000,2.619415,3.778735,4.686697,1.16,5.50,12.00,41.0,2.940935,2.722174,4.173095,3.880495,5.45658,4.757603,36.5166,2.003017,1.902307,2.033331,1.930361,23.13209,-0.303451,2.004782,1.93259,2.173407,2.071169,3.1928,4.493388,5.08369,3.003213,4.243009,4.610582,1.890303,2.207561,1.821665,2.

In [33]:
#pd.set_option('display.max_rows', None)
pd.reset_option('display.max_rows')
print("Shape of df:", df.shape)

missing_counts = df.isnull().sum().sort_values(ascending=False)
print("\nMissing values per column:")
print(missing_counts)

missing_perc = (df.isnull().mean() * 100).sort_values(ascending=False)
print("\nPercentage missing per column:")
print(missing_perc)

high_nan_cols = missing_perc[missing_perc > 95].index.tolist()
print("\nColumns with >95% missing values:", high_nan_cols)

print(f"\nTotal columns: {df.shape[1]}")
print(f"Columns with no missing: {(missing_counts==0).sum()}")
print(f"Columns with >50% missing: {(missing_perc > 50).sum()}")

Shape of df: (7901, 266)

Missing values per column:
Date                    0
AwayTeam_Leicester      0
AwayTeam_Bournemouth    0
AwayTeam_Brentford      0
AwayTeam_Brighton       0
                       ..
AvgH                    0
AvgD                    0
AvgA                    0
Max>2.5                 0
home_away_odds_ratio    0
Length: 266, dtype: int64

Percentage missing per column:
Date                    0.0
AwayTeam_Leicester      0.0
AwayTeam_Bournemouth    0.0
AwayTeam_Brentford      0.0
AwayTeam_Brighton       0.0
                       ... 
AvgH                    0.0
AvgD                    0.0
AvgA                    0.0
Max>2.5                 0.0
home_away_odds_ratio    0.0
Length: 266, dtype: float64

Columns with >95% missing values: []

Total columns: 266
Columns with no missing: 266
Columns with >50% missing: 0


In [36]:
df.to_csv("../../data/processed/first_engineered_betting_features.csv", index=False)

In [37]:
df.columns.tolist()

['Date',
 'FTHG',
 'FTAG',
 'FTR',
 'HTHG',
 'HTAG',
 'HTR',
 'Referee',
 'HS',
 'AS',
 'HST',
 'AST',
 'HC',
 'AC',
 'HF',
 'AF',
 'HY',
 'AY',
 'HR',
 'AR',
 'B365H',
 'B365D',
 'B365A',
 'B365>2.5',
 'B365<2.5',
 'B365AHH',
 'B365AHA',
 'B365AH',
 'VCH',
 'VCD',
 'VCA',
 'BSH',
 'BSD',
 'BSA',
 'BWH',
 'BWD',
 'BWA',
 'GBH',
 'GBD',
 'GBA',
 'GB>2.5',
 'GB<2.5',
 'GBAHH',
 'GBAHA',
 'GBAH',
 'IWH',
 'IWD',
 'IWA',
 'LBH',
 'LBD',
 'LBA',
 'LBAHH',
 'LBAHA',
 'LBAH',
 'PSH',
 'PSD',
 'PSA',
 'P>2.5',
 'P<2.5',
 'PAHH',
 'PAHA',
 'SOH',
 'SOD',
 'SOA',
 'SBH',
 'SBD',
 'SBA',
 'SJH',
 'SJD',
 'SJA',
 'WHH',
 'WHD',
 'WHA',
 'Bb1X2',
 'BbMxH',
 'BbAvH',
 'BbMxD',
 'BbAvD',
 'BbMxA',
 'BbAvA',
 'BbOU',
 'BbMx>2.5',
 'BbAv>2.5',
 'BbMx<2.5',
 'BbAv<2.5',
 'BbAH',
 'BbAHh',
 'BbMxAHH',
 'BbAvAHH',
 'BbMxAHA',
 'BbAvAHA',
 'MaxH',
 'MaxD',
 'MaxA',
 'AvgH',
 'AvgD',
 'AvgA',
 'Max>2.5',
 'Max<2.5',
 'Avg>2.5',
 'Avg<2.5',
 'MaxAHH',
 'MaxAHA',
 'AvgAHH',
 'AvgAHA',
 'AHh',
 'Season',
 'Yea